# COMP8240 — Final VADER Research Evaluation

**One executable computational record: replication, generalisation, human annotation and LLM assessment.**

Research questions:
1. **RQ1:** Which original VADER results can be reproduced, and how does the same implementation perform on mapped GoEmotions and contemporary YouTube comments?
2. **RQ2:** How reliably do the two humans and three saved LLM judges assign sentiment, relative to a clearly defined human reference?
3. **RQ3:** What paired performance differences and observable disagreement patterns are supported, given sampling, annotation and provenance limitations?

This notebook recomputes results from real input files. The **minimally adapted 2014 implementation is the primary replicated system throughout**; package 3.3.2 is explicitly a version sensitivity analysis. LLMs are alternative annotators/classifiers, not automatically valid judges or independent gold standards. No live API calls, new annotation or model-based adjudication are performed.

**Reading guide:** sections 1–5 establish inputs and methods; 6–8 reproduce and extend VADER; 9–15 assess human/machine judgments; 16–19 quantify uncertainty and inspect errors; 20–21 consolidate findings and provenance. All reported intervals are conditional on the observed corpus construction, not population-level guarantees.

**Artifact scope:** this notebook contains the complete analysis. Reusable command-line modules are also supplied under `src/`; they use the same estimands and are checked against this independent notebook execution. Input datasets remain external, while tables and figures render inline. This local notebook contains private research outputs and short redacted excerpts, and inherits the existing YouTube source refresh/delete lifecycle.

## 1. Assessment alignment and research design

The supplied final-report brief requests original-work replication, construction of new data, results, annotation agreement where applicable, the role of LLM judging, ethics and a runnable public code repository. It specifies ACL-style LaTeX, at most 10 main-body pages excluding references, and a deadline of **6 November 2026, 23:55 Sydney time**. This notebook supports the computational component; it does not replace the written report.

The proposal considered several benchmarks and social platforms. The implemented extension is **GoEmotions plus 200 YouTube comments from four selected videos**. Those scope changes must be explained rather than presented as completed proposal commitments. Commenter ages are unknown; no Gen Z population claim is justified. Local lecture-specific ethics material and an adjudication protocol were not supplied.

The original study [Hutto & Gilbert, 2014](https://doi.org/10.1609/icwsm.v8i1.14550) evaluates sentiment intensity and three-class polarity in four domains. Its Table 4 reports *overall F1*, which is not ordinary accuracy and has no fully specified averaging recipe in the available paper. The local historical-source PDF, rather than the currently access-blocked publisher page, was inspected for that table and the threshold description.

## 2. Reproducibility settings and dependencies

Run top to bottom using a Python environment with the pinned direct dependencies below. Existing project environment: `.venv/bin/python`. No secret files or network credentials are read. Libraries are not installed automatically. Set `COMP8240_ROOT` if the notebook is moved outside the project tree.

The nine human rating levels are **ordinal**. Averaging and continuous agreement calculations additionally assume equal intervals; these are transparent project choices. LLM intensity scale compatibility is supported by the researcher-supplied shared rubric, using the researcher-supplied shared −4..+4 rubric; exact historical provider settings remain unavailable. Classification remains the primary LLM comparison.

In [ ]:
from pathlib import Path
import os, sys, json, csv, ast, re, hashlib, platform, sqlite3, itertools, warnings
from datetime import datetime, timezone
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import scipy
from scipy.stats import pearsonr, spearmanr, binomtest
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             f1_score, cohen_kappa_score)
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from IPython.display import display, Markdown
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
get_ipython().run_line_magic('matplotlib', 'inline')

candidates = [Path(os.environ['COMP8240_ROOT'])] if 'COMP8240_ROOT' in os.environ else [Path.cwd(), *Path.cwd().parents]
ROOT = next((p.resolve() for p in candidates if (p/'data/goemotions/processed/all.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Set COMP8240_ROOT to the project containing data/goemotions/processed/all.csv')
SEED, BOOTSTRAPS = 8240, 2000
LABELS = ['negative', 'neutral', 'positive']
CLASS_INDEX = {v:i for i,v in enumerate(LABELS)}
SYSTEMS = ['VADER 2014', 'VADER 3.3.2', 'ChatGPT', 'Claude', 'Gemini']
PRIMARY_SYSTEMS = ['VADER 2014', 'ChatGPT', 'Claude', 'Gemini']
PALETTE = dict(zip(SYSTEMS, ['#40576d','#7b8790','#2b7d78','#af7139','#835f93']))
sns.set_theme(style='whitegrid', context='notebook', font_scale=.95)
plt.rcParams.update({'figure.dpi':110, 'savefig.dpi':220, 'axes.spines.top':False,
                     'axes.spines.right':False, 'figure.max_open_warning':30})
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 35)
PINS = {'vaderSentiment': '3.3.2', 'numpy': '2.5.4', 'pandas': '3.1.1', 'scipy': '1.18.1', 'scikit-learn': '1.9.1', 'matplotlib': '3.10.7', 'seaborn': '0.13.2', 'statsmodels': '0.15.0', 'PyYAML': '6.0.3', 'nbformat': '5.11.1', 'nbclient': '0.11.0', 'ipykernel': '7.4.0'}
environment = pd.DataFrame([{'package':p,'pinned':v,'installed':metadata.version(p)} for p,v in PINS.items()])
display(environment)
print('Python:', platform.python_version(), '| Platform:', platform.platform())
print('Execution UTC:', datetime.now(timezone.utc).isoformat(), '| seed:', SEED, '| bootstrap repetitions:', BOOTSTRAPS)
print('Install direct dependencies with: python -m pip install ' + ' '.join(f'{p}=={v}' for p,v in PINS.items()))
assert metadata.version('vaderSentiment') == '3.3.2', 'Modern version sensitivity requires package 3.3.2'

def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def note(text):
    display(Markdown(text))

INPUT_HASHES = {}
def track(path):
    path = Path(path)
    if path.is_file(): INPUT_HASHES[str(path.relative_to(ROOT))] = sha(path)
    return path

def load_csv(relative, **kwargs):
    p=track(ROOT/relative)
    return pd.read_csv(p, keep_default_na=False, **kwargs)

print('Project:', ROOT)

## 3. Input inventory and provenance audit

The inventory is calculated from file contents, not historical README claims. It distinguishes ground-truth aggregates, per-rater artifacts, processed data, candidate pools, annotation exports and saved predictions. Archived revisions are provenance inputs, not additional observations. The root README and older three-label annotation guide describe an earlier stage; the app's nine-level intensity guide is the current task rubric.

Dataset sources: original VADER release and GoEmotions sources are recorded in the local acquisition manifest. GoEmotions' emotion-to-polarity mapping is **project-derived**, not a replacement for the original emotion taxonomy [Demszky et al., 2020](https://aclanthology.org/2020.acl-main.372/). YouTube was acquired through an official API client using bounded recent-comment frames, then eligibility filters and unique-text sampling; seven records were later replaced. The sample is therefore quality-amended rather than an untouched simple random sample.

In [ ]:
ORIGINAL_DIR=ROOT/'data/original_vader'
ORIGINAL_FILES={'Tweets':'tweets_GroundTruth.txt','Movie':'movieReviewSnippets_GroundTruth.txt',
                'Amazon':'amazonReviewSnippets_GroundTruth.txt','NYT':'nytEditorialSnippets_GroundTruth.txt'}
EXPECTED_ORIGINAL={'Tweets':4200,'Movie':10605,'Amazon':3708,'NYT':5190}
originals={}
inventory=[]
for name,filename in ORIGINAL_FILES.items():
    p=track(ORIGINAL_DIR/filename)
    records=[]
    with p.open(encoding='utf-8', newline='') as f:
        for line in f:
            sid, score, text=line.rstrip('\r\n').split('\t',2)
            records.append((sid,float(score),text))
    df=pd.DataFrame(records,columns=['sample_id','human_intensity_raw','original_text'])
    assert len(df)==EXPECTED_ORIGINAL[name] and df.sample_id.is_unique
    assert df.original_text.str.strip().ne('').all() and df.human_intensity_raw.between(-4,4).all()
    df['dataset']=name
    originals[name]=df
    inventory.append({'input':name+' ground truth','path':str(p.relative_to(ROOT)), 'format':'headerless TSV',
                      'rows':len(df),'columns':', '.join(df.columns[:3])})

for rel in ['data/goemotions/processed/all.csv','data/goemotions/processed/excluded.csv',
            'data/youtube/sampled/youtube_final_200.csv','data/youtube/annotations/annotator_1_annotations.csv',
            'data/youtube/annotations/annotator_2_annotations.csv','data/youtube/annotations/combined_human_annotations.csv',
            'data/youtube/llm/predictions/chatgpt.csv','data/youtube/llm/predictions/claude.csv','data/youtube/llm/predictions/gemini.csv']:
    d=load_csv(rel)
    inventory.append({'input':Path(rel).stem,'path':rel,'format':'CSV','rows':len(d),'columns':', '.join(d.columns)})
for p in sorted((ROOT/'data/youtube/raw').glob('video_*/comments_retrieved.csv')):
    d=load_csv(str(p.relative_to(ROOT)))
    inventory.append({'input':p.parent.name+' candidate frame','path':str(p.relative_to(ROOT)),
                      'format':'CSV','rows':len(d),'columns':', '.join(d.columns)})
inventory=pd.DataFrame(inventory)
display(inventory)

source_manifest=yaml.safe_load(track(ROOT/'data/provenance/upstream_sources.yaml').read_text())
provenance=[]
for d in source_manifest.get('datasets',[]):
    provenance.append({k:d.get(k,'unavailable') for k in ['name','source','version','licence']})
display(pd.DataFrame(provenance))

# Inventory supporting work without loading secrets or treating old outputs as truth.
support=[]
for p in sorted(ROOT.rglob('*.ipynb')):
    if '.venv' in p.parts or p.name=='COMP8240_VADER_Final_Research_Evaluation.ipynb': continue
    j=json.loads(p.read_text())
    support.append({'notebook':str(p.relative_to(ROOT)), 'code_cells':sum(c['cell_type']=='code' for c in j['cells']),
                    'executed_cells':sum(c.get('execution_count') is not None for c in j['cells'] if c['cell_type']=='code')})
display(pd.DataFrame(support))
doc_paths=['README.md','docs/human_annotation_guidelines.md','docs/human_annotation_guidelines.md',
           'src/evaluation/run_all_metrics.py','src/replication/legacy_vader.py','config/preprocessing.yaml',
           'data/original_vader/_README_.txt',
           'data/original_vader/vader_icwsm2014_final.pdf',
           'reports/data_processing/goemotions_preprocessing_summary.json','data/youtube/metadata/youtube_preprocessing_summary.json',
           'data/youtube/metadata/youtube_final_sampling_report.md','data/youtube/metadata/youtube_final_sampling_validation.json',
           'docs/youtube_access_and_retention.md']
display(pd.DataFrame([{'supporting_input':r,'exists':(ROOT/r).is_file(),
                       'bytes':track(ROOT/r).stat().st_size if (ROOT/r).is_file() else None} for r in doc_paths]))
print('No adjudication file, lecture pack or historical LLM protocol has been established by this inventory.')

### 3.1 Original-release internal consistency

Per-rater files are audited independently using stable IDs and their actual rating lists. Their IDs and mean scores must not be assumed to match the text-bearing ground-truth files. The primary replication continues to use the latter. We neither repair source annotations nor substitute a more favourable reference. Anonymous rating lists do not establish persistent cross-item rater identities, so they cannot support a new individual-rater replication without further evidence.

In [ ]:
rater_audit=[]
for name,filename in ORIGINAL_FILES.items():
    p=track(ORIGINAL_DIR/filename.replace('GroundTruth','anonDataRatings'))
    rows=[]
    with p.open(encoding='utf-8',newline='') as f:
        for line in f:
            sid, mean, sd, values=line.rstrip('\r\n').split('\t',3)
            ratings=ast.literal_eval(values)
            assert ratings and all(-4<=v<=4 for v in ratings)
            rows.append((sid,float(mean),float(sd),len(ratings),float(np.mean(ratings))))
    rd=pd.DataFrame(rows,columns=['sample_id','stored_mean','stored_sd','rating_count','list_mean'])
    assert rd.sample_id.is_unique
    matched=originals[name][['sample_id','human_intensity_raw']].merge(rd,on='sample_id',validate='one_to_one')
    rater_audit.append({'dataset':name,'text_records':len(originals[name]),'rater_records':len(rd),'matched_ids':len(matched),
        'ground_truth_mean_mismatches':int((abs(matched.human_intensity_raw-matched.stored_mean)>1e-6).sum()),
        'stored_vs_list_mean_mismatches':int((abs(rd.stored_mean-rd.list_mean)>1e-6).sum()),
        'minimum_ratings':rd.rating_count.min(),'maximum_ratings':rd.rating_count.max()})
    inventory.loc[len(inventory)]={'input':name+' anonymous ratings','path':str(p.relative_to(ROOT)),
        'format':'headerless TSV','rows':len(rd),'columns':'sample_id, stored mean, SD, anonymous rating list'}
rater_audit=pd.DataFrame(rater_audit)
display(rater_audit)
note('**Interpretation:** rating-list counts vary and some source artifacts do not align. These are concrete reproducibility limitations of the available release, not proof that a particular alternative score file is the authors’ intended evaluation reference.')

## 4. Measurement protocol and metric catalogue

**Separate reference rules:** original-corpus means are divided by 4, then thresholded at ±0.05 as an explicit interpretation of the paper's normalized-score statement. Raw-human ±0.05 and sign-only rules are sensitivity diagnostics, never tuned. For the current human task, negative/zero/positive raw ratings define polarity; the averaged raw rating uses the same sign rule. This is motivated by the app's explicit zero-neutral anchor, not by applying VADER cutoffs to raw human scores. With these integer ratings, it happens to agree with ±0.05 after normalization. GoEmotions uses its documented emotion mapping. Saved LLM polarity labels are evaluated directly; scores are not used to overwrite them.

| Measure | Input and purpose | Why selected / interpretation | Limitations / source |
|---|---|---|---|
| Accuracy / exact agreement | Matched nominal labels; fraction equal | Transparent item-level success or pairwise agreement | Majority prevalence can dominate; equality with humans is not objective truth. [Hutto & Gilbert](https://doi.org/10.1609/icwsm.v8i1.14550) |
| Per-class precision | Predicted and reference labels; TP/(TP+FP) | Reliability of a system's class assignments | Reference dependent; zero when prediction denominator is zero by declared convention. Same source |
| Per-class recall | Labels; TP/(TP+FN) | Coverage of each reference class | Class-specific denominators and support matter. Same source |
| Per-class F1 | Labels; harmonic mean of precision and recall | Summarises class retrieval performance | Does not count true negatives; not ordinary accuracy. Same source |
| Macro P/R/F1 | Unweighted mean across fixed three classes | Gives neutral and minority negative classes equal weight | Sensitive to rare classes; macro F1 is not F1 of macro P/R. Same source; project aggregation choice |
| Weighted F1 | Per-class F1 weighted by reference support | Shows prevalence-sensitive alternative | Can conceal minority failures; not interchangeable with macro F1. Same source; aggregation stated here |
| Cohen κ | Two aligned nominal-label sequences | Agreement after marginal chance correction | Depends on prevalence/marginals; no universal 0.65 acceptance rule. [Cohen, 1960](https://doi.org/10.1177/001316446002000104), [Artstein & Poesio](https://aclanthology.org/J08-4004/) |
| Linear / quadratic weighted κ | Two ordinal nine-level score sequences | Gives partial credit, penalising larger gaps linearly/quadratically | Weighting and score order must be defensible; primary categorical κ stays unweighted. [Cohen, 1968](https://doi.org/10.1037/h0026256) |
| Pearson r | Paired numeric intensities with variation | Linear association, as in original VADER evaluation | High r does not ensure agreement, calibration or causality; equal-interval assumption |
| Spearman ρ | Paired numeric/ordinal intensities | Rank association suitable as ordinal sensitivity | Ties and range restrictions matter; monotonic association is not agreement |
| MAE / RMSE | Scores on a declared shared range | Typical absolute / larger-error-sensitive distance | Scale dependent; /4 scaling does not establish construct equivalence |
| Lin CCC | Numeric paired scores on common interval interpretation | Penalises mean and scale disagreement as well as imperfect correlation | Exploratory for ordinal ratings and nonlinear VADER compound; [Lin, 1989](https://pubmed.ncbi.nlm.nih.gov/2720055/) |
| Percentile bootstrap CI | Matched item resamples | Quantifies conditional sampling variability | Not uncertainty about annotation bias, selection or missing clusters; [Dror et al., 2018](https://aclanthology.org/P18-1128/) |
| Paired Δmacro-F1 CI | Identical resampled items for two systems | Direct uncertainty on observed performance difference | Conditional, correlated-video caveat; same source |
| Exact McNemar test | Paired correctness indicators, discordant counts | Tests marginal accuracy equality on common items | Item independence is an assumption; does not test F1. Same source |

Pearson, Spearman and error measures follow the original intensity-evaluation framework; continuous-versus-ordinal caveats are informed by the agreement survey. Fleiss κ and mixed human–machine pooled reliability are omitted: there are exactly two independent human roles, and pooling unlike evaluator types would change the estimand. Krippendorff α is optional rather than necessary for this complete two-rater design; it is not added merely to increase the metric count.

In [ ]:
def vader_polarity(scores):
    a=np.asarray(scores,dtype=float)
    return np.where(a>=.05,'positive',np.where(a<=-.05,'negative','neutral'))

def human_polarity(scores):
    a=np.asarray(scores,dtype=float)
    return np.where(a>0,'positive',np.where(a<0,'negative','neutral'))

def encoded(labels):
    a=np.asarray(labels)
    assert set(a)<=set(LABELS), f'Unknown sentiment classes: {set(a)-set(LABELS)}'
    return np.array([CLASS_INDEX[x] for x in a],dtype=np.int64)

def ccc(a,b):
    a,b=np.asarray(a,dtype=float),np.asarray(b,dtype=float)
    den=np.var(a)+np.var(b)+(np.mean(a)-np.mean(b))**2
    return float(2*np.mean((a-a.mean())*(b-b.mean()))/den) if den>0 else np.nan

def intensity_metrics(a,b):
    a,b=np.asarray(a,dtype=float),np.asarray(b,dtype=float)
    assert len(a)==len(b) and np.isfinite(a).all() and np.isfinite(b).all()
    variable=len(a)>1 and np.ptp(a)>0 and np.ptp(b)>0
    return {'pearson_r':float(pearsonr(a,b).statistic) if variable else np.nan,
            'spearman_rho':float(spearmanr(a,b).statistic) if variable else np.nan,
            'mae':float(np.mean(abs(a-b))), 'rmse':float(np.sqrt(np.mean((a-b)**2))), 'ccc':ccc(a,b)}

def evaluate(y,p,human_score=None,system_score=None):
    y,p=np.asarray(y),np.asarray(p)
    assert len(y)==len(p)>0
    encoded(y);encoded(p)
    r=classification_report(y,p,labels=LABELS,output_dict=True,zero_division=0)
    m={'n':len(y),'accuracy':float(np.mean(y==p)), 'macro_precision':r['macro avg']['precision'],
       'macro_recall':r['macro avg']['recall'],'macro_f1':r['macro avg']['f1-score'],
       'weighted_f1':r['weighted avg']['f1-score'],'micro_f1':float(np.mean(y==p)),
       'cohen_kappa':float(cohen_kappa_score(y,p,labels=LABELS)) if len(set(y)|set(p))>1 else np.nan}
    if human_score is not None: m.update(intensity_metrics(human_score,system_score))
    return m,pd.DataFrame(r).T.loc[LABELS,['precision','recall','f1-score','support']],confusion_matrix(y,p,labels=LABELS)

def fast_class(y,p,k=3):
    cm=np.bincount(k*y+p,minlength=k*k).reshape(k,k).astype(float)
    n=cm.sum();tp=np.diag(cm);rs=cm.sum(axis=1);cs=cm.sum(axis=0)
    f=np.divide(2*tp,rs+cs,out=np.zeros(k),where=(rs+cs)>0)
    pe=float(rs@cs/n**2);acc=float(tp.sum()/n)
    return acc,float(f.mean()),(acc-pe)/(1-pe) if pe<1 else np.nan

def weighted_kappa_fast(a,b,quadratic=True):
    cm=np.bincount(9*(a.astype(int)+4)+(b.astype(int)+4),minlength=81).reshape(9,9).astype(float)
    distance=abs(np.arange(9)[:,None]-np.arange(9)[None,:]).astype(float)
    if quadratic: distance=distance**2
    expected=np.outer(cm.sum(1),cm.sum(0))/cm.sum()
    den=(expected*distance).sum()
    return 1-float((cm*distance).sum()/den) if den else np.nan

def bootstrap_indices(n,groups=None,b=BOOTSTRAPS,seed=SEED):
    rng=np.random.default_rng(seed)
    pools=[np.arange(n)] if groups is None else [np.flatnonzero(np.asarray(groups)==g) for g in pd.unique(groups)]
    return np.concatenate([rng.choice(pool,size=(b,len(pool)),replace=True) for pool in pools],axis=1)

def percentile_ci(values):
    values=np.asarray(values,dtype=float);finite=values[np.isfinite(values)]
    return (float(np.quantile(finite,.025)),float(np.quantile(finite,.975)),len(values)-len(finite)) if len(finite) else (np.nan,np.nan,len(values))

def bootstrap_evaluation(y,p,indices,score_a=None,score_b=None):
    y,p=encoded(y),encoded(p)
    columns=['accuracy','macro_f1','cohen_kappa']
    if score_a is not None:
        a,b=np.asarray(score_a,dtype=float),np.asarray(score_b,dtype=float)
        columns+=['pearson_r','ccc','mae']
    out=[]
    for ix in indices:
        row=list(fast_class(y[ix],p[ix]))
        if score_a is not None:
            aa,bb=a[ix],b[ix]
            r=float(np.corrcoef(aa,bb)[0,1]) if np.ptp(aa)>0 and np.ptp(bb)>0 else np.nan
            row += [r,ccc(aa,bb),float(np.mean(abs(aa-bb)))]
        out.append(row)
    return pd.DataFrame(out,columns=columns)

def add_intervals(row,draws):
    for key in draws:
        lo,hi,bad=percentile_ci(draws[key])
        row.update({key+'_ci_low':lo,key+'_ci_high':hi,key+'_degenerate_draws':bad})
    return row

def confusion_plot(cm,title,ax,normalise=False):
    shown=np.divide(cm,cm.sum(1,keepdims=True),out=np.zeros_like(cm,dtype=float),where=cm.sum(1,keepdims=True)>0) if normalise else cm
    sns.heatmap(shown,annot=True,fmt='.2f' if normalise else 'd',cmap='Blues',cbar=False,
                xticklabels=LABELS,yticklabels=LABELS,ax=ax,vmin=0,vmax=1 if normalise else None)
    ax.set(title=title,xlabel='Predicted / comparison polarity',ylabel='Reference polarity')
    ax.tick_params(axis='y',rotation=0)

def close_figure(fig,caption):
    fig.tight_layout()
    plt.show()
    plt.close(fig)
    note(caption)

# Independent checks against library metrics and analytically known cases.
assert list(vader_polarity([-.05,-.04999,0,.04999,.05]))==['negative','neutral','neutral','neutral','positive']
assert list(human_polarity([-1,0,1]))==LABELS
assert np.isclose(ccc([0,1,2],[0,1,2]),1)
assert not np.isclose(ccc([0,1,2],[1,2,3]),1) # perfect correlation is not perfect concordance
rng=np.random.default_rng(SEED)
for _ in range(15):
    a,b=rng.integers(0,3,200),rng.integers(0,3,200)
    acc,mf,k=fast_class(a,b)
    assert np.isclose(acc,accuracy_score(a,b)) and np.isclose(mf,f1_score(a,b,labels=[0,1,2],average='macro',zero_division=0))
    assert np.isclose(k,cohen_kappa_score(a,b,labels=[0,1,2]))
a,b=rng.integers(-4,5,200),rng.integers(-4,5,200)
assert np.isclose(weighted_kappa_fast(a,b),cohen_kappa_score(a,b,labels=list(range(-4,5)),weights='quadratic'))
print('Boundary, correlation/concordance, macro-F1, kappa and ordinal-kappa checks passed.')

## 5. Systems and statistical design

The historical source is embedded in the following cell so the analysis does not depend on an external analysis script. Its original notice is retained. The compatibility changes are Python 2 syntax conversion, removal of `reload`, explicit lexicon path and byte-preserving Latin-1 decoding of a lexicon that is not valid UTF-8. The demonstration block is omitted; sentiment rules and lexical values are unchanged. Python 3 Unicode token behaviour still prevents a claim of bit-for-bit equivalence to an unavailable Python 2 runtime.

The package implementation receives the same unmodified text and uses its own lexicon and built-in emoji handling. No externally expanded emojis or lowercase text are substituted.

**Uncertainty plan (project-specific):** 2,000 percentile bootstrap draws, seed 8240, preserving prediction/reference pairs. YouTube draws resample comments **within each source video**, retaining the observed allocation. Other corpus intervals are item-level conditional summaries. Items within one review/video may be dependent; stratification does not eliminate within-video dependence. With only four videos, reliable population-level cluster inference is not available. Intervals exclude annotation-protocol, source-selection and model-generation uncertainty. Undefined correlation/kappa draws are omitted with their counts reported; absent-class F1 uses a fixed three-class zero convention. McNemar p-values are exploratory under an item-independence assumption, with Holm correction across the declared comparison family. No human-equivalence decision is based on an arbitrary κ cutoff. All bootstrap intervals are pointwise 95% intervals, not simultaneous family-wise intervals; only the declared McNemar p-value family receives Holm adjustment.

In [ ]:
# Compatibility adaptation of bundled 2014 source; no rule or lexicon edits.
# Python 2 tuple-lambda/map converted; reload removed; lexicon path explicit.
# Latin-1 preserves each byte of the legacy lexicon (not valid UTF-8).
#!/usr/bin/python
# coding: utf-8 
'''
Created on July 04, 2013
@author: C.J. Hutto

Citation Information

If you use any of the VADER sentiment analysis tools 
(VADER sentiment lexicon or Python code for rule-based sentiment 
analysis engine) in your work or research, please cite the paper. 
For example:

  Hutto, C.J. & Gilbert, E.E. (2014). VADER: A Parsimonious Rule-based Model for 
  Sentiment Analysis of Social Media Text. Eighth International Conference on 
  Weblogs and Social Media (ICWSM-14). Ann Arbor, MI, June 2014.
'''

import math, re, sys, fnmatch, string


from pathlib import Path
f = ROOT / 'data/original_vader/vader_sentiment_lexicon.txt' # empirically derived valence ratings for words, emoticons, slang, swear words, acronyms/initialisms
word_valence_dict = dict([(w_m[0], float(w_m[1])) for w_m in [
            wmsr.strip().split('\t')[0:2] for wmsr in open(f, encoding='latin-1') ]])

# for removing punctuation
regex_remove_punctuation = re.compile('[%s]' % re.escape(string.punctuation))

def sentiment(text):
    """
    Returns a float for sentiment strength based on the input text.
    Positive values are positive valence, negative value are negative valence.
    """
    wordsAndEmoticons = str(text).split() #doesn't separate words from adjacent punctuation (keeps emoticons & contractions)
    text_mod = regex_remove_punctuation.sub('', text) # removes punctuation (but loses emoticons & contractions)
    wordsOnly = str(text_mod).split()
    # get rid of empty items or single letter "words" like 'a' and 'I' from wordsOnly
    for word in wordsOnly:
        if len(word) <= 1:
            wordsOnly.remove(word)    
    # now remove adjacent & redundant punctuation from [wordsAndEmoticons] while keeping emoticons and contractions
    puncList = [".", "!", "?", ",", ";", ":", "-", "'", "\"", 
                "!!", "!!!", "??", "???", "?!?", "!?!", "?!?!", "!?!?"] 
    for word in wordsOnly:
        for p in puncList:
            pword = p + word
            x1 = wordsAndEmoticons.count(pword)
            while x1 > 0:
                i = wordsAndEmoticons.index(pword)
                wordsAndEmoticons.remove(pword)
                wordsAndEmoticons.insert(i, word)
                x1 = wordsAndEmoticons.count(pword)
            
            wordp = word + p
            x2 = wordsAndEmoticons.count(wordp)
            while x2 > 0:
                i = wordsAndEmoticons.index(wordp)
                wordsAndEmoticons.remove(wordp)
                wordsAndEmoticons.insert(i, word)
                x2 = wordsAndEmoticons.count(wordp)
    # get rid of residual empty items or single letter "words" like 'a' and 'I' from wordsAndEmoticons
    for word in wordsAndEmoticons:
        if len(word) <= 1:
            wordsAndEmoticons.remove(word)
    
    # remove stopwords from [wordsAndEmoticons]
    #stopwords = [str(word).strip() for word in open('stopwords.txt')]
    #for word in wordsAndEmoticons:
    #    if word in stopwords:
    #        wordsAndEmoticons.remove(word)
    
    # check for negation
    negate = ["aint", "arent", "cannot", "cant", "couldnt", "darent", "didnt", "doesnt",
              "ain't", "aren't", "can't", "couldn't", "daren't", "didn't", "doesn't",
              "dont", "hadnt", "hasnt", "havent", "isnt", "mightnt", "mustnt", "neither",
              "don't", "hadn't", "hasn't", "haven't", "isn't", "mightn't", "mustn't",
              "neednt", "needn't", "never", "none", "nope", "nor", "not", "nothing", "nowhere", 
              "oughtnt", "shant", "shouldnt", "uhuh", "wasnt", "werent",
              "oughtn't", "shan't", "shouldn't", "uh-uh", "wasn't", "weren't",  
              "without", "wont", "wouldnt", "won't", "wouldn't", "rarely", "seldom", "despite"]
    def negated(list, nWords=[], includeNT=True):
        nWords.extend(negate)
        for word in nWords:
            if word in list:
                return True
        if includeNT:
            for word in list:
                if "n't" in word:
                    return True
        if "least" in list:
            i = list.index("least")
            if i > 0 and list[i-1] != "at":
                return True
        return False
        
    def normalize(score, alpha=15):
        # normalize the score to be between -1 and 1 using an alpha that approximates the max expected value 
        normScore = score/math.sqrt( ((score*score) + alpha) )
        return normScore
    
    def wildCardMatch(patternWithWildcard, listOfStringsToMatchAgainst):
        listOfMatches = fnmatch.filter(listOfStringsToMatchAgainst, patternWithWildcard)
        return listOfMatches
        
    
    def isALLCAP_differential(wordList):
        countALLCAPS= 0
        for w in wordList:
            if str(w).isupper(): 
                countALLCAPS += 1
        cap_differential = len(wordList) - countALLCAPS
        if cap_differential > 0 and cap_differential < len(wordList):
            isDiff = True
        else: isDiff = False
        return isDiff
    isCap_diff = isALLCAP_differential(wordsAndEmoticons)
    
    b_incr = 0.293 #(empirically derived mean sentiment intensity rating increase for booster words)
    b_decr = -0.293
    # booster/dampener 'intensifiers' or 'degree adverbs' http://en.wiktionary.org/wiki/Category:English_degree_adverbs
    booster_dict = {"absolutely": b_incr, "amazingly": b_incr, "awfully": b_incr, "completely": b_incr, "considerably": b_incr, 
                    "decidedly": b_incr, "deeply": b_incr, "effing": b_incr, "enormously": b_incr, 
                    "entirely": b_incr, "especially": b_incr, "exceptionally": b_incr, "extremely": b_incr,
                    "fabulously": b_incr, "flipping": b_incr, "flippin": b_incr, 
                    "fricking": b_incr, "frickin": b_incr, "frigging": b_incr, "friggin": b_incr, "fully": b_incr, "fucking": b_incr, 
                    "greatly": b_incr, "hella": b_incr, "highly": b_incr, "hugely": b_incr, "incredibly": b_incr, 
                    "intensely": b_incr, "majorly": b_incr, "more": b_incr, "most": b_incr, "particularly": b_incr, 
                    "purely": b_incr, "quite": b_incr, "really": b_incr, "remarkably": b_incr, 
                    "so": b_incr,  "substantially": b_incr, 
                    "thoroughly": b_incr, "totally": b_incr, "tremendously": b_incr, 
                    "uber": b_incr, "unbelievably": b_incr, "unusually": b_incr, "utterly": b_incr, 
                    "very": b_incr, 
                    
                    "almost": b_decr, "barely": b_decr, "hardly": b_decr, "just enough": b_decr, 
                    "kind of": b_decr, "kinda": b_decr, "kindof": b_decr, "kind-of": b_decr,
                    "less": b_decr, "little": b_decr, "marginally": b_decr, "occasionally": b_decr, "partly": b_decr, 
                    "scarcely": b_decr, "slightly": b_decr, "somewhat": b_decr, 
                    "sort of": b_decr, "sorta": b_decr, "sortof": b_decr, "sort-of": b_decr}
    sentiments = []
    for item in wordsAndEmoticons:
        v = 0
        i = wordsAndEmoticons.index(item)
        if (i < len(wordsAndEmoticons)-1 and str(item).lower() == "kind" and \
           str(wordsAndEmoticons[i+1]).lower() == "of") or str(item).lower() in booster_dict:
            sentiments.append(v)
            continue
        item_lowercase = str(item).lower() 
        if  item_lowercase in word_valence_dict:
            #get the sentiment valence
            v = float(word_valence_dict[item_lowercase])
            
            #check if sentiment laden word is in ALLCAPS (while others aren't)
            c_incr = 0.733 #(empirically derived mean sentiment intensity rating increase for using ALLCAPs to emphasize a word)
            if str(item).isupper() and isCap_diff:
                if v > 0: v += c_incr
                else: v -= c_incr
            
            #check if the preceding words increase, decrease, or negate/nullify the valence
            def scalar_inc_dec(word, valence):
                scalar = 0.0
                word_lower = str(word).lower()
                if word_lower in booster_dict:
                    scalar = booster_dict[word_lower]
                    if valence < 0: scalar *= -1
                    #check if booster/dampener word is in ALLCAPS (while others aren't)
                    if str(word).isupper() and isCap_diff:
                        if valence > 0: scalar += c_incr
                        else:  scalar -= c_incr
                return scalar
            n_scalar = -0.74
            if i > 0 and str(wordsAndEmoticons[i-1]).lower() not in word_valence_dict:
                s1 = scalar_inc_dec(wordsAndEmoticons[i-1], v)
                v = v+s1
                if negated([wordsAndEmoticons[i-1]]): v = v*n_scalar
            if i > 1 and str(wordsAndEmoticons[i-2]).lower() not in word_valence_dict:
                s2 = scalar_inc_dec(wordsAndEmoticons[i-2], v)
                if s2 != 0: s2 = s2*0.95
                v = v+s2
                # check for special use of 'never' as valence modifier instead of negation
                if wordsAndEmoticons[i-2] == "never" and (wordsAndEmoticons[i-1] == "so" or wordsAndEmoticons[i-1] == "this"): 
                    v = v*1.5                    
                # otherwise, check for negation/nullification
                elif negated([wordsAndEmoticons[i-2]]): v = v*n_scalar
            if i > 2 and str(wordsAndEmoticons[i-3]).lower() not in word_valence_dict:
                s3 = scalar_inc_dec(wordsAndEmoticons[i-3], v)
                if s3 != 0: s3 = s3*0.9
                v = v+s3
                # check for special use of 'never' as valence modifier instead of negation
                if wordsAndEmoticons[i-3] == "never" and \
                   (wordsAndEmoticons[i-2] == "so" or wordsAndEmoticons[i-2] == "this") or \
                   (wordsAndEmoticons[i-1] == "so" or wordsAndEmoticons[i-1] == "this"):
                    v = v*1.25
                # otherwise, check for negation/nullification
                elif negated([wordsAndEmoticons[i-3]]): v = v*n_scalar
                
                # check for special case idioms using a sentiment-laden keyword known to SAGE
                special_case_idioms = {"the shit": 3, "the bomb": 3, "bad ass": 1.5, "yeah right": -2, 
                                       "cut the mustard": 2, "kiss of death": -1.5, "hand to mouth": -2}
                # future work: consider other sentiment-laden idioms
                #other_idioms = {"back handed": -2, "blow smoke": -2, "blowing smoke": -2, "upper hand": 1, "break a leg": 2, 
                #                "cooking with gas": 2, "in the black": 2, "in the red": -2, "on the ball": 2,"under the weather": -2}
                onezero = "{} {}".format(str(wordsAndEmoticons[i-1]), str(wordsAndEmoticons[i]))
                twoonezero = "{} {}".format(str(wordsAndEmoticons[i-2]), str(wordsAndEmoticons[i-1]), str(wordsAndEmoticons[i]))
                twoone = "{} {}".format(str(wordsAndEmoticons[i-2]), str(wordsAndEmoticons[i-1]))
                threetwoone = "{} {} {}".format(str(wordsAndEmoticons[i-3]), str(wordsAndEmoticons[i-2]), str(wordsAndEmoticons[i-1]))
                threetwo = "{} {}".format(str(wordsAndEmoticons[i-3]), str(wordsAndEmoticons[i-2]))                    
                if onezero in special_case_idioms: v = special_case_idioms[onezero]
                elif twoonezero in special_case_idioms: v = special_case_idioms[twoonezero]
                elif twoone in special_case_idioms: v = special_case_idioms[twoone]
                elif threetwoone in special_case_idioms: v = special_case_idioms[threetwoone]
                elif threetwo in special_case_idioms: v = special_case_idioms[threetwo]
                if len(wordsAndEmoticons)-1 > i:
                    zeroone = "{} {}".format(str(wordsAndEmoticons[i]), str(wordsAndEmoticons[i+1]))
                    if zeroone in special_case_idioms: v = special_case_idioms[zeroone]
                if len(wordsAndEmoticons)-1 > i+1:
                    zeroonetwo = "{} {}".format(str(wordsAndEmoticons[i]), str(wordsAndEmoticons[i+1]), str(wordsAndEmoticons[i+2]))
                    if zeroonetwo in special_case_idioms: v = special_case_idioms[zeroonetwo]
                
                # check for booster/dampener bi-grams such as 'sort of' or 'kind of'
                if threetwo in booster_dict or twoone in booster_dict:
                    v = v+b_decr
            
            # check for negation case using "least"
            if i > 1 and str(wordsAndEmoticons[i-1]).lower() not in word_valence_dict \
                and str(wordsAndEmoticons[i-1]).lower() == "least":
                if (str(wordsAndEmoticons[i-2]).lower() != "at" and str(wordsAndEmoticons[i-2]).lower() != "very"):
                    v = v*n_scalar
            elif i > 0 and str(wordsAndEmoticons[i-1]).lower() not in word_valence_dict \
                and str(wordsAndEmoticons[i-1]).lower() == "least":
                v = v*n_scalar
        sentiments.append(v) 
            
    # check for modification in sentiment due to contrastive conjunction 'but'
    if 'but' in wordsAndEmoticons or 'BUT' in wordsAndEmoticons:
        try: bi = wordsAndEmoticons.index('but')
        except: bi = wordsAndEmoticons.index('BUT')
        for s in sentiments:
            si = sentiments.index(s)
            if si < bi: 
                sentiments.pop(si)
                sentiments.insert(si, s*0.5)
            elif si > bi: 
                sentiments.pop(si)
                sentiments.insert(si, s*1.5) 
                
    if sentiments:                      
        sum_s = float(sum(sentiments))
        #print sentiments, sum_s
        
        # check for added emphasis resulting from exclamation points (up to 4 of them)
        ep_count = str(text).count("!")
        if ep_count > 4: ep_count = 4
        ep_amplifier = ep_count*0.292 #(empirically derived mean sentiment intensity rating increase for exclamation points)
        if sum_s > 0:  sum_s += ep_amplifier
        elif  sum_s < 0: sum_s -= ep_amplifier
        
        # check for added emphasis resulting from question marks (2 or 3+)
        qm_count = str(text).count("?")
        qm_amplifier = 0
        if qm_count > 1:
            if qm_count <= 3: qm_amplifier = qm_count*0.18
            else: qm_amplifier = 0.96
            if sum_s > 0:  sum_s += qm_amplifier
            elif  sum_s < 0: sum_s -= qm_amplifier

        compound = normalize(sum_s)
        
        # want separate positive versus negative sentiment scores
        pos_sum = 0.0
        neg_sum = 0.0
        neu_count = 0
        for sentiment_score in sentiments:
            if sentiment_score > 0:
                pos_sum += (float(sentiment_score) +1) # compensates for neutral words that are counted as 1
            if sentiment_score < 0:
                neg_sum += (float(sentiment_score) -1) # when used with math.fabs(), compensates for neutrals
            if sentiment_score == 0:
                neu_count += 1
        
        if pos_sum > math.fabs(neg_sum): pos_sum += (ep_amplifier+qm_amplifier)
        elif pos_sum < math.fabs(neg_sum): neg_sum -= (ep_amplifier+qm_amplifier)
        
        total = pos_sum + math.fabs(neg_sum) + neu_count
        pos = math.fabs(pos_sum / total)
        neg = math.fabs(neg_sum / total)
        neu = math.fabs(neu_count / total)
        
    else:
        compound = 0.0; pos = 0.0; neg = 0.0; neu = 0.0
        
    s = {"neg" : round(neg, 3), 
         "neu" : round(neu, 3),
         "pos" : round(pos, 3),
         "compound" : round(compound, 4)}
    return s



legacy_sentiment=sentiment
modern=SentimentIntensityAnalyzer()
track(ORIGINAL_DIR/'vader_sentiment_lexicon.txt')
track(ORIGINAL_DIR/'vaderSentiment.py')
lex_a,lex_b=set(word_valence_dict),set(modern.lexicon)
lexicon_comparison=pd.DataFrame([{'legacy_entries':len(lex_a),'modern_entries':len(lex_b),
    'modern_only':len(lex_b-lex_a),'legacy_only':len(lex_a-lex_b),
    'changed_shared_valences':sum(word_valence_dict[k]!=modern.lexicon[k] for k in lex_a&lex_b)}])
display(lexicon_comparison)
import vaderSentiment.vaderSentiment as modern_module
MODEL_HASHES={p.name:sha(p) for p in [Path(modern_module.__file__),Path(modern_module.__file__).parent/'vader_lexicon.txt',Path(modern_module.__file__).parent/'emoji_utf8_lexicon.txt']}
print('No sentiment model was trained or thresholds tuned.')

## 6. Experiment A — Original VADER replication

All four complete text-bearing corpora are evaluated, with no train/test split introduced for a fixed rule system. Each gets its original aggregate human score, normalized intensity, operational reference label and both implementations' predictions. The historical code is the primary replication. Published values are transcribed from Table 4; *paper overall F1* remains a separate column from local macro/weighted F1. Confidence intervals quantify variability of this release only, not differences from an unknown exact 2014 environment.

In [ ]:
PAPER=pd.DataFrame([
    ['Tweets',4200,.881,.99,.94,.96],['Movie',10605,.451,.70,.55,.61],
    ['Amazon',3708,.565,.70,.55,.63],['NYT',5190,.492,.69,.49,.55]],
    columns=['dataset','paper_n','paper_r','paper_precision','paper_recall','paper_overall_f1'])
original_rows=[]; original_per_class={}; original_cms={}; original_boot={}
for name,df in originals.items():
    df['human_intensity']=df.human_intensity_raw/4
    df['human_gold_label']=vader_polarity(df.human_intensity)
    df['gold_label_source']='released aggregate rating /4, operational +/-0.05'
    indices=bootstrap_indices(len(df))
    for system,scorer in [('VADER 2014',legacy_sentiment),('VADER 3.3.2',modern.polarity_scores)]:
        df[system+'_score']=[scorer(t)['compound'] for t in df.original_text]
        df[system+'_label']=vader_polarity(df[system+'_score'])
        m,pc,cm=evaluate(df.human_gold_label,df[system+'_label'],df.human_intensity,df[system+'_score'])
        draws=bootstrap_evaluation(df.human_gold_label,df[system+'_label'],indices,df.human_intensity,df[system+'_score'])
        original_boot[(name,system)]=draws
        original_rows.append(add_intervals({'dataset':name,'system':system,**m},draws))
        original_per_class[(name,system)]=pc;original_cms[(name,system)]=cm
original_results=pd.DataFrame(original_rows).merge(PAPER,on='dataset',validate='many_to_one')
assert (original_results.n==original_results.paper_n).all()
original_results['delta_r']=original_results.pearson_r-original_results.paper_r
display(original_results[['dataset','system','n','pearson_r','pearson_r_ci_low','pearson_r_ci_high',
    'paper_r','delta_r','accuracy','macro_precision','macro_recall','macro_f1','weighted_f1','paper_overall_f1']].round(4))
print('Paper F1 is not accuracy; its averaging is not assumed to equal local macro F1.')

In [ ]:
sensitivity=[]
for name,df in originals.items():
    rules={'normalized +/-0.05 (primary)':vader_polarity(df.human_intensity),
           'raw +/-0.05 (diagnostic)':vader_polarity(df.human_intensity_raw),
           'raw sign (diagnostic)':human_polarity(df.human_intensity_raw)}
    for rule,gold in rules.items():
        m,_,_=evaluate(gold,df['VADER 2014_label'])
        r=classification_report(gold,df['VADER 2014_label'],labels=LABELS,output_dict=True,zero_division=0)
        sensitivity.append({'dataset':name,'human_rule':rule,'neutral_count':int((gold=='neutral').sum()),
            'accuracy':m['accuracy'],'macro_f1':m['macro_f1'],'weighted_f1':m['weighted_f1'],
            'weighted_precision':r['weighted avg']['precision'],'weighted_recall':r['weighted avg']['recall']})
sensitivity=pd.DataFrame(sensitivity)
display(sensitivity.round(4))
version_changes=pd.DataFrame([{'dataset':n,'changed_compound':int((d['VADER 2014_score']!=d['VADER 3.3.2_score']).sum()),
    'changed_polarity':int((d['VADER 2014_label']!=d['VADER 3.3.2_label']).sum())} for n,d in originals.items()])
display(version_changes)
fig,ax=plt.subplots(figsize=(8.6,4))
xx=np.arange(len(ORIGINAL_FILES))
for offset,system in [(-.22,'VADER 2014'),(0,'VADER 3.3.2')]:
    d=original_results[original_results.system==system].set_index('dataset').loc[list(ORIGINAL_FILES)]
    ax.bar(xx+offset,d.pearson_r,width=.22,label=system,color=PALETTE[system])
ax.bar(xx+.22,PAPER.set_index('dataset').loc[list(ORIGINAL_FILES)].paper_r,width=.22,label='Published Pearson r',color='#a6aaa8')
ax.set(xticks=xx,xticklabels=list(ORIGINAL_FILES),xlabel='Original benchmark corpus',ylabel='Pearson correlation r',ylim=(0,1),title='Original-corpus intensity correlation: published and recomputed')
ax.legend(fontsize=9)
close_figure(fig,'**Figure 1.** Source: released VADER ground-truth corpora and Table 4. Tweet and NYT historical correlations agree with the paper after rounding; movie and Amazon differ. This supports partial replication, not exact reproduction of every statistic.')

tw=original_results.query("dataset=='Tweets' and system=='VADER 2014'").iloc[0]
ny=original_results.query("dataset=='NYT' and system=='VADER 2014'").iloc[0]
rawtw=sensitivity.query("dataset=='Tweets' and human_rule=='raw +/-0.05 (diagnostic)'").iloc[0]
note(f'**Replication finding:** historical tweet r={tw.pearson_r:.4f} and NYT r={ny.pearson_r:.4f} reproduce their published values to three decimals. Under the alternate raw-human threshold, tweet weighted F1={rawtw.weighted_f1:.4f} rounds to .96 and weighted recall={rawtw.weighted_recall:.4f}, but weighted precision={rawtw.weighted_precision:.4f} still differs from .99. Threshold construction and averaging explain much of the headline classification gap without establishing the original recipe. Data-artifact inconsistencies, version effects and the missing original evaluation driver remain distinct unresolved factors.')

## 7. Experiment B — GoEmotions classification extension

The released agreement-filtered split files contain 54,263 records, not the approximate full-release count in the paper. Their original multi-label emotions are mapped using the existing project configuration. Ambiguous emotions, mixed positive/negative emotions and neutral-plus-other labels are excluded. We verify retained IDs, unchanged text, original splits and every mapped target against raw labels. We report all splits, but the **official test split is the primary extension estimate**. No intensity correlation is calculated for nominal sentiment targets. [Demszky et al.](https://aclanthology.org/2020.acl-main.372/) motivates using the fine-grained corpus, not this particular project mapping.

In [ ]:
cfg=yaml.safe_load(track(ROOT/'config/preprocessing.yaml').read_text())
mapping=cfg['goemotions_mapping']
display(pd.DataFrame([{'mapped_group':k,'original_emotions':', '.join(v)} for k,v in mapping.items()]))
go=load_csv('data/goemotions/processed/all.csv',dtype={'id':str})
excluded_go=load_csv('data/goemotions/processed/excluded.csv',dtype={'id':str})
raw_go=[]
emotions=track(ROOT/cfg['goemotions_root']/'emotions.txt').read_text().splitlines()
for split,filename in cfg['splits'].items():
    p=track(ROOT/cfg['goemotions_root']/filename)
    with p.open(encoding='utf-8',newline='') as f:
        for text,labels,sid in csv.reader(f,delimiter='\t',quoting=csv.QUOTE_NONE):
            raw_go.append((sid,text,labels,split))
raw_go=pd.DataFrame(raw_go,columns=['id','raw_text','raw_labels','raw_split'])
assert raw_go.id.is_unique and go.id.is_unique and excluded_go.id.is_unique
assert len(go)+len(excluded_go)==len(raw_go)
assert not set(go.id)&set(excluded_go.id)
assert set(go.id)|set(excluded_go.id)==set(raw_go.id)
verified=go.merge(raw_go,on='id',validate='one_to_one')
assert (verified.text_original==verified.raw_text).all() and (verified.original_split==verified.raw_split).all()
assert (verified.original_emotion_labels.astype(str)==verified.raw_labels).all()
def mapped_target(names):
    s=set(names)
    if s&set(mapping['ambiguous']) or (s&set(mapping['positive']) and s&set(mapping['negative'])) or ('neutral' in s and len(s)>1):return None
    for label in LABELS:
        if s and s<=set(mapping[label]):return label
    return None
recomputed=verified.raw_labels.map(lambda ids:mapped_target([emotions[int(i)] for i in ids.split(',')]))
assert (recomputed==verified.mapped_sentiment).all()
go=go.rename(columns={'id':'sample_id','text_original':'original_text','mapped_sentiment':'human_gold_label'})
go['dataset']='GoEmotions';go['gold_label_source']='project emotion-to-polarity mapping'
for system,scorer in [('VADER 2014',legacy_sentiment),('VADER 3.3.2',modern.polarity_scores)]:
    go[system+'_score']=[scorer(t)['compound'] for t in go.original_text]
    go[system+'_label']=vader_polarity(go[system+'_score'])
go_rows=[];go_pc={};go_cm={};go_boot={}
for split in ['train','validation','test','all_descriptive']:
    subset=go if split=='all_descriptive' else go[go.original_split==split]
    for system in SYSTEMS[:2]:
        m,pc,cm=evaluate(subset.human_gold_label,subset[system+'_label'])
        row={'split':split,'system':system,**m}
        if split=='test':
            draws=bootstrap_evaluation(subset.human_gold_label,subset[system+'_label'],bootstrap_indices(len(subset)))
            row=add_intervals(row,draws);go_boot[system]=draws
        go_rows.append(row);go_pc[(split,system)]=pc;go_cm[(split,system)]=cm
        
go_results=pd.DataFrame(go_rows)
display(go_results.round(4))
print('Raw / retained / excluded:',len(raw_go),len(go),len(excluded_go))
print('Exclusion primary reasons:',excluded_go.exclusion_reason.value_counts().to_dict())
cross_groups=raw_go.groupby('raw_text').raw_split.nunique().gt(1).sum()
print('Exact-text groups spanning official splits:',int(cross_groups))
note('**Interpretation:** the mapping is reproduced for every retained example, but it changes the target construct and class distribution. Exact-text overlap across official splits remains documented; no training or tuning is performed, so no fitted model learns from those overlaps. Existing emotion labels are not continuous sentiment ground truth.')

## 8. Experiment C — YouTube data preparation and alignment

The four source videos are confounded with category, creator and audience. Collection used up to ten pages of 100 recent comments per video, producing a bounded frame rather than all comments or a platform-random sample. Eligibility screening and unique-text sampling further restrict the population. Seven later replacements were recorded; both current annotators rated the replacement IDs. Only original text is used for scoring; no fake `cleaned_text` field is introduced.

The notebook checks source text and IDs, both annotation exports, combined export, and the local database when available. CSV exports are the portable primary annotation inputs; a database mismatch stops evaluation instead of silently choosing a version. The database is accessed read-only without inspecting credentials or sessions.

In [ ]:
sample=load_csv('data/youtube/sampled/youtube_final_200.csv',dtype={'sample_id':str,'comment_id':str})
assert len(sample)==200 and sample.sample_id.is_unique and sample.comment_id.is_unique
assert sample.comment_text_original.str.replace(r'\s+',' ',regex=True).str.strip().str.casefold().is_unique
assert sample.groupby('video_id').size().eq(50).all() and sample.video_id.nunique()==4
sampling=json.loads(track(ROOT/'data/youtube/metadata/youtube_final_sampling_validation.json').read_text())
deadline=datetime.fromisoformat(sampling['retention_deadline'])
assert datetime.now(timezone.utc)<deadline, 'Not computable: source snapshot has expired; refresh sources and derivatives first'
raw_youtube=pd.concat([load_csv(str(p.relative_to(ROOT)),dtype={'comment_id':str}) for p in sorted((ROOT/'data/youtube/raw').glob('video_*/comments_retrieved.csv'))],ignore_index=True)
assert raw_youtube.comment_id.is_unique
raw_text_column=next(c for c in ['comment_text_original','comment_text','text_original','text'] if c in raw_youtube.columns)
text_lookup=raw_youtube.set_index('comment_id')[raw_text_column]
assert sample.comment_id.isin(text_lookup.index).all()
assert (sample.comment_text_original.to_numpy()==text_lookup.loc[sample.comment_id].to_numpy()).all()
revision=load_csv('data/youtube/revisions/quality_revision_20261103/replacement_audit.csv')
assert set(revision.new_sample_id)<=set(sample.sample_id) and not set(revision.old_sample_id)&set(sample.sample_id)

yt=sample.rename(columns={'comment_text_original':'original_text','video_category':'domain','video_id':'source_video_id'}).copy()
yt['dataset']='YouTube'
annotation_tables={}; join_audit=[]
for n in [1,2]:
    a=load_csv(f'data/youtube/annotations/annotator_{n}_annotations.csv',dtype={'sample_id':str})
    assert a.sample_id.is_unique and set(a.sample_id)==set(yt.sample_id)
    assert (a.annotator_id==f'annotator_{n}').all()
    assert (a.set_index('sample_id').comment_text_original.loc[yt.sample_id].to_numpy()==yt.original_text.to_numpy()).all()
    a['score']=pd.to_numeric(a.intensity_score,errors='coerce')
    valid=a.annotation_status.eq('completed') & a.score.notna() & a.score.between(-4,4) & a.score.eq(a.score.round())
    join_audit.append({'source':f'annotator_{n}','input_rows':len(a),'matched_ids':len(a),
                       'completed_valid':int(valid.sum()),'excluded_missing_or_unresolved':int((~valid).sum())})
    a.loc[~valid,'score']=np.nan
    annotation_tables[n]=a
    yt=yt.merge(a[['sample_id','score']].rename(columns={'score':f'human_{n}_raw'}),on='sample_id',validate='one_to_one')
combined=load_csv('data/youtube/annotations/combined_human_annotations.csv',dtype={'sample_id':str}).set_index('sample_id')
assert set(combined.index)==set(yt.sample_id)
for n in [1,2]:
    combined_values=pd.to_numeric(combined.loc[yt.sample_id,f'annotator_{n}_intensity'],errors='coerce').to_numpy()
    assert np.allclose(combined_values,yt[f'human_{n}_raw'],equal_nan=True)
submissions=[]
dbp=ROOT/'annotation-app/database/annotations.db'
if dbp.is_file():
    with sqlite3.connect(dbp.resolve().as_uri()+'?mode=ro',uri=True) as conn:
        conn.execute('BEGIN')
        dbtexts=dict(conn.execute('select sample_id,text from samples'))
        dbratings={(a,s):v for a,s,v in conn.execute("select annotator_id,sample_id,score from annotations where status='completed'")}
        submissions=[{'annotator':a,'finalized_flag':bool(done),'finalized_at':stamp} for a,done,stamp in conn.execute("select role,submission_complete,finalized_at from users where role in ('annotator_1','annotator_2')")]
    assert set(dbtexts)==set(yt.sample_id)
    assert all(dbtexts[r.sample_id]==r.original_text for r in yt.itertuples())
    for n in [1,2]:
        assert all(dbratings.get((f'annotator_{n}',r.sample_id))==getattr(r,f'human_{n}_raw') for r in yt.itertuples() if pd.notna(getattr(r,f'human_{n}_raw')))

display(pd.DataFrame(join_audit))
display(pd.DataFrame(submissions))
print('Candidate frame:',len(raw_youtube),'| selected:',len(yt),'| recorded replacements:',len(revision))
print('Source expiry UTC:',deadline.isoformat())
quality_rows=[]
for folder in sorted((ROOT/'data/youtube/raw').glob('video_*')):
    name=folder.name
    original=load_csv(str((folder/'comments_retrieved.csv').relative_to(ROOT)))
    retained=load_csv(f'data/youtube/processed/{name}/youtube_comments_cleaned.csv')
    pending=load_csv(f'data/youtube/processed/{name}/youtube_comments_review_required.csv')
    excluded=load_csv(f'data/youtube/excluded/{name}/youtube_comments_excluded.csv')
    assert len(retained)+len(pending)+len(excluded)==len(original)
    selected=sample[sample.video_id.isin(original.video_id.unique())]
    assert set(selected.comment_id)<=set(retained.comment_id)
    quality_rows.append({'source_folder':name,'original_candidates':len(original),
        'retained_candidates':len(retained),'review_required':len(pending),'excluded':len(excluded),'selected':len(selected)})
display(pd.DataFrame(quality_rows))
print('Pending-review candidates were not silently treated as eligible or assigned neutral labels.')
note('**Status interpretation:** saved completion and the finalisation flag are separate fields. A complete score export is usable as a dated analysis snapshot, but an unset finalisation flag is reported rather than overwritten. Human ratings and submission status are never modified here.')

## 9. Human–human reliability

Calculate nominal polarity κ and raw agreement before constructing a combined reference. Weighted κ is applied to the ordered nine-level intensity scores, not to arbitrary numeric codes for nominal labels. Correlation, absolute distance and CCC answer different questions. Human bootstrap draws preserve the same paired comments and video strata. Individual ratings remain unchanged; agreement does not prove correct interpretation or verify that annotators worked independently.

In [ ]:
paired=yt.dropna(subset=['human_1_raw','human_2_raw']).copy()
assert len(paired)>0,'Not computable: no paired completed human scores'
a,b=paired.human_1_raw.to_numpy(),paired.human_2_raw.to_numpy()
h1,h2=human_polarity(a),human_polarity(b)
HH,HH_PC,HH_CM=evaluate(h1,h2,a/4,b/4)
HH.update(exact_intensity_agreement=float(np.mean(a==b)),linear_weighted_kappa=float(cohen_kappa_score(a,b,labels=list(range(-4,5)),weights='linear')),
          quadratic_weighted_kappa=float(cohen_kappa_score(a,b,labels=list(range(-4,5)),weights='quadratic')),
          raw_intensity_mae=float(np.mean(abs(a-b))),raw_intensity_rmse=float(np.sqrt(np.mean((a-b)**2))))
hh_indices=bootstrap_indices(len(paired),paired.source_video_id)
hh_draws=bootstrap_evaluation(h1,h2,hh_indices,a/4,b/4)
hh_draws['quadratic_weighted_kappa']=[weighted_kappa_fast(a[ix],b[ix]) for ix in hh_indices]
HH=add_intervals(HH,hh_draws)
human_agreement_table=pd.DataFrame([HH])
display(human_agreement_table.T.rename(columns={0:'estimate'}).round(4))
human_class_agreement=pd.DataFrame({'class':LABELS,'both_assign':np.diag(HH_CM),
    'annotator_1_support':HH_CM.sum(1),'annotator_2_support':HH_CM.sum(0),
    'annotator_1_only':HH_CM.sum(1)-np.diag(HH_CM),'annotator_2_only':HH_CM.sum(0)-np.diag(HH_CM)})
display(human_class_agreement)
print('Historical kappa .693 delta:',HH['cohen_kappa']-.693,'| weighted kappa .791 delta:',HH['quadratic_weighted_kappa']-.791)
print('Actual polarity disagreements:',int(np.sum(h1!=h2)),'| historical count: 33')
fig,axes=plt.subplots(1,2,figsize=(10,4.3))
confusion_plot(HH_CM,f'Human polarity agreement (n={len(paired)})',axes[0])
axes[0].set(xlabel='Annotator 2 polarity',ylabel='Annotator 1 polarity')
axes[1].scatter(a,b,alpha=.35,s=35,color='#40576d')
axes[1].plot([-4,4],[-4,4],color='gray',linestyle='--',label='Exact agreement')
axes[1].set(xlabel='Annotator 1 intensity (−4 to +4)',ylabel='Annotator 2 intensity (−4 to +4)',title='Human intensity: nine ordered rating levels')
axes[1].legend(fontsize=8)
close_figure(fig,'**Figure 2.** Source: matched independent-role human exports. The diagonal shows exact agreement; departures reflect real score differences. Overlapping points obscure frequency in the scatter, so the numerical exact-agreement and error measures are also reported. These ratings alone cannot verify the independence of the annotation process.')

## 10. Human reference construction and disagreement audit

**Primary currently available reference:** average both completed human raw scores, with positive/zero/negative polarity. It is an operational averaged reference, **not an adjudicated gold standard**. Opposing sentiments may cancel to zero; sensitivity analyses below expose this issue. No annotator is automatically privileged. If a complete independently adjudicated file is supplied at `data/youtube/annotations/youtube_adjudicated_labels.csv`, its actual labels become primary after provenance and status checks. Required columns: `sample_id`, `human_gold_label`, `adjudicator_id`, `adjudication_timestamp`, `adjudication_status` with status `adjudicated`. These checks validate structure, not the substantive independence of adjudication; no model output should be shown to human adjudicators beforehand.

A partial adjudication file is analysed only on its matched adjudicated subset, separately, rather than silently mixing references. No adjudicated labels are manufactured from agreement or averaged scores.

In [ ]:
yt['human_mean_raw']=yt[['human_1_raw','human_2_raw']].mean(axis=1,skipna=False)
yt['human_mean_intensity']=yt.human_mean_raw/4
for n in [1,2]:
    yt[f'human_{n}_intensity']=yt[f'human_{n}_raw']/4
    yt[f'human_{n}_label']=pd.Series(np.where(yt[f'human_{n}_raw'].notna(),human_polarity(yt[f'human_{n}_raw']),None),index=yt.index)
yt['human_mean_label']=np.where(yt.human_mean_raw.notna(),human_polarity(yt.human_mean_raw),None)
yt['human_gold_label']=yt.human_mean_label
yt['gold_label_source']='averaged independent-role intensities; sign polarity; not adjudicated'
yt['adjudication_status']='not available'
adjudication_path=ROOT/'data/youtube/annotations/youtube_adjudicated_labels.csv'
adjudicated=None
if adjudication_path.is_file():
    ad=load_csv(str(adjudication_path.relative_to(ROOT)),dtype={'sample_id':str})
    required={'sample_id','human_gold_label','adjudicator_id','adjudication_timestamp','adjudication_status'}
    assert required<=set(ad.columns) and ad.sample_id.is_unique
    assert set(ad.sample_id)<=set(yt.sample_id)
    assert ad.human_gold_label.isin(LABELS).all() and ad.adjudication_status.eq('adjudicated').all()
    assert ad.adjudicator_id.str.strip().ne('').all() and pd.to_datetime(ad.adjudication_timestamp,errors='coerce',utc=True).notna().all()
    adjudicated=ad
    if len(ad)==len(yt):
        yt['human_gold_label']=ad.set_index('sample_id').loc[yt.sample_id,'human_gold_label'].to_numpy()
        yt['gold_label_source']='supplied independently adjudicated labels; provenance fields checked'
        yt['adjudication_status']='adjudicated'
    else: note(f'**Partial adjudication:** {len(ad)} records; primary full-cohort reference remains averaged, with separate matched-subset evaluation later.')
else:
    note('**Not computable from available data:** adjudicated-reference performance. No actual adjudication records were found.')

human_disagreements=yt.loc[yt.human_1_raw.notna()&yt.human_2_raw.notna(),
    ['sample_id','domain','human_1_raw','human_2_raw','human_1_label','human_2_label','human_mean_raw']].copy()
human_disagreements['absolute_gap']=abs(human_disagreements.human_1_raw-human_disagreements.human_2_raw)
human_disagreements['polarity_disagreement']=human_disagreements.human_1_label!=human_disagreements.human_2_label
human_disagreements=human_disagreements.query('absolute_gap>0').sort_values(['polarity_disagreement','absolute_gap'],ascending=False)
display(human_disagreements.head(20))
print('All intensity disagreements:',len(human_disagreements),'| polarity disagreements:',int(human_disagreements.polarity_disagreement.sum()),
      '| score gaps >=3:',int(human_disagreements.absolute_gap.ge(3).sum()))
reference_summary=yt.groupby('gold_label_source').size().rename('n').reset_index()
display(reference_summary)

## 11. LLM judging protocol and saved-output validation

The files identify provider families as ChatGPT, Claude and Gemini, but not exact model snapshots. **Unavailable:** exact model identifiers, inference dates, temperature/top-p, original API schema, retry logs, proof of blindness to human ratings, and repeated generations. File modification times are not inference dates. Saved justifications are model claims, not verified causal explanations or hidden internal reasoning.

Observed output structure is validated: ID, integer sentiment score, named polarity, numeric confidence, boolean uncertainty and justification. It is not presented as a recovered historical prompt. Labels are evaluated directly; scores under the researcher-supplied shared −4..+4 rubric are divided by 4 only for explicitly conditional intensity diagnostics. Confidence is self-reported, not a calibrated correctness probability. IDs can establish alignment but cannot establish the original prompt text or input-text hash at inference.

LLM-judge papers motivate validation, not automatic transfer of reported performance. [Zheng et al., 2023](https://arxiv.org/abs/2306.05685) studies conversational preferences; [Liu et al., 2023](https://aclanthology.org/2023.emnlp-main.153/) studies NLG evaluation. Neither is a validation of these sentiment files. [Bavaresco et al., 2025](https://aclanthology.org/2025.acl-short.20/) finds task/model variability, supporting local comparisons with human judgments. Here the LLMs directly annotate sentiment; they do not blindly rate a VADER explanation.

In [ ]:
track(ROOT/'prompts/supplied_judge_prompt.md')
track(ROOT/'prompts/protocol.json')
llm_protocol=[];llm_validation=[]
for model,slug in [('ChatGPT','chatgpt'),('Claude','claude'),('Gemini','gemini')]:
    d=load_csv(f'data/youtube/llm/predictions/{slug}.csv',dtype={'sample_id':str})
    required={'sample_id','sentiment_score','sentiment_label','confidence','uncertain','justification'}
    assert required<=set(d.columns) and d.sample_id.is_unique
    assert set(d.sample_id)==set(yt.sample_id), f'{model}: missing or extra IDs'
    d['score']=pd.to_numeric(d.sentiment_score,errors='coerce')
    d['confidence_value']=pd.to_numeric(d.confidence,errors='coerce')
    assert d.score.notna().all() and d.score.between(-4,4).all() and d.score.eq(d.score.round()).all()
    assert d.sentiment_label.isin(LABELS).all() and (d.sentiment_label.to_numpy()==human_polarity(d.score)).all()
    assert d.confidence_value.notna().all() and d.confidence_value.between(0,1).all()
    boolean=d.uncertain.astype(str).str.lower()
    assert boolean.isin(['true','false']).all()
    llm_validation.append({'system':model,'input_rows':len(d),'matched':len(yt),'missing':0,'invalid_or_failed_saved_rows':0,
                           'uncertain_rows':int(boolean.eq('true').sum()),'observed_score_min':d.score.min(),'observed_score_max':d.score.max()})
    yt=yt.merge(d[['sample_id','sentiment_label','score','confidence_value']].rename(columns={
        'sentiment_label':model+'_label','score':model+'_raw','confidence_value':model+'_confidence'}),on='sample_id',validate='one_to_one')
    yt[model+'_score']=yt[model+'_raw']/4
    llm_protocol.append({'system':model,'exact_model_id':'unavailable','prompts':'researcher-supplied shared prompt; role placement unavailable','inference_date':'unavailable',
                         'temperature_top_p':'unavailable','observed_saved_schema':', '.join(sorted(required)),
                         'repeat_generations':'unavailable','annotation_blinding_evidence':'unavailable'})
# Optional JSON copy is a consistency check, not an extra run.
p=ROOT/'data/youtube/llm/historical_outputs/chatgpt_saved_prediction_objects.json'
if p.is_file():
    js=pd.DataFrame(json.loads(track(p).read_text())).set_index('sample_id')
    assert set(js.index)==set(yt.sample_id)
    assert (js.loc[yt.sample_id,'sentiment_label'].to_numpy()==yt.ChatGPT_label.to_numpy()).all()
    assert np.allclose(pd.to_numeric(js.loc[yt.sample_id,'sentiment_score']),yt.ChatGPT_raw)
display(pd.DataFrame(llm_protocol))
display(pd.DataFrame(llm_validation))
note('No malformed saved rows were admitted. Zero observed failures in saved files does not establish that no generation failures or selective retries occurred historically. Repeated-run instability is **not computable from available data**.')
for system,scorer in [('VADER 2014',legacy_sentiment),('VADER 3.3.2',modern.polarity_scores)]:
    yt[system+'_score']=[scorer(t)['compound'] for t in yt.original_text]
    yt[system+'_label']=vader_polarity(yt[system+'_score'])
assert len(yt)==200 and yt.sample_id.is_unique
# Exact identity field retained privately; displayed audits use sample IDs and categories.
print('Aligned YouTube frame:',yt.shape,'| source IDs, texts, human scores and all five system outputs validated.')

## 12. Master YouTube evaluation — human reference versus VADER and LLMs

Every system is evaluated on the **same matched human-reference cohort**, with the same label set and video-stratified resamples. The averaged human intensity remains an auxiliary score reference even if adjudicated categorical labels are supplied, since no adjudicated intensity is invented. Standard macro P/R/F1 and per-class scores use a fixed three-label order. The primary replicated VADER and modern version are explicitly separate rows.

In [ ]:
evaluation_yt=yt.dropna(subset=['human_gold_label','human_mean_intensity']).reset_index(drop=True)
assert len(evaluation_yt)>0
Y=evaluation_yt.human_gold_label.to_numpy()
YT_INDICES=bootstrap_indices(len(evaluation_yt),evaluation_yt.source_video_id)
yt_rows=[];yt_per_class={};yt_confusions={};yt_boot={}
for system in SYSTEMS:
    m,pc,cm=evaluate(Y,evaluation_yt[system+'_label'],evaluation_yt.human_mean_intensity,evaluation_yt[system+'_score'])
    draws=bootstrap_evaluation(Y,evaluation_yt[system+'_label'],YT_INDICES,evaluation_yt.human_mean_intensity,evaluation_yt[system+'_score'])
    yt_boot[system]=draws;yt_per_class[system]=pc;yt_confusions[system]=cm
    yt_rows.append(add_intervals({'system':system,'reference':evaluation_yt.gold_label_source.iloc[0],**m},draws))
master_results=pd.DataFrame(yt_rows)
display(master_results[['system','n','accuracy','accuracy_ci_low','accuracy_ci_high','macro_f1','macro_f1_ci_low','macro_f1_ci_high',
    'weighted_f1','macro_precision','macro_recall','cohen_kappa','cohen_kappa_ci_low','cohen_kappa_ci_high',
    'pearson_r','spearman_rho','mae','rmse','ccc']].round(4))
all_system_per_class=pd.concat(yt_per_class,names=['system','reference_class']).reset_index()
display(all_system_per_class.round(4))
print('Degenerate draws by statistic:')
display(master_results[['system']+[c for c in master_results if c.endswith('_degenerate_draws')]])
majority_label=pd.Series(Y).value_counts().idxmax()
majority_metrics,_,_=evaluate(Y,np.repeat(majority_label,len(Y)))
print('Post-hoc majority-label sanity check:',majority_label,'accuracy',round(majority_metrics['accuracy'],4),'macro F1',round(majority_metrics['macro_f1'],4))
note('The majority row is a post-hoc descriptive sanity check, not a trained or pre-registered competing model. LLM score-distance/CCC columns depend on the researcher-supplied intensity rubric and its unverified historical role placement and an equal-interval approximation; categorical comparisons are better supported. A high correlation with a rescaled reference does not establish good calibration.')

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4.4))
for ax,metric,title in [(axes[0],'accuracy','Polarity accuracy'),(axes[1],'macro_f1','Three-class macro F1')]:
    xx=np.arange(len(SYSTEMS));d=master_results.set_index('system').loc[SYSTEMS]
    ax.bar(xx,d[metric],color=[PALETTE[s] for s in SYSTEMS])
    ax.errorbar(xx,d[metric],yerr=np.vstack([d[metric]-d[metric+'_ci_low'],d[metric+'_ci_high']-d[metric]]),fmt='none',ecolor='black',capsize=3)
    ax.set(xticks=xx,xticklabels=SYSTEMS,ylabel=metric.replace('_',' ').title()+' (0–1)',xlabel='Evaluated system',ylim=(0,1),title=title+f' (n={len(Y)})')
    ax.tick_params(axis='x',rotation=30)
close_figure(fig,'**Figure 3.** Source: saved judge outputs and human exports on identical YouTube records. Error bars are 95% video-stratified item-bootstrap percentile intervals. Apparent rank order alone does not establish superiority; paired-difference intervals and annotation-reference sensitivity follow. VADER 3.3.2 is a version sensitivity row.')
fig,axes=plt.subplots(2,3,figsize=(12,7.5))
for ax,system in zip(axes.flat,SYSTEMS):
    confusion_plot(yt_confusions[system],system+f' (n={len(Y)})',ax)
axes.flat[-1].axis('off')
close_figure(fig,'**Figure 4.** Counts against the same human reference. Row supports are shared, making systematic neutral-versus-polar errors visible. Confusion matrices describe disagreements with this operational reference, not independently adjudicated truth.')
fig,axes=plt.subplots(1,2,figsize=(9.5,4))
for ax,normalise in zip(axes,[False,True]):
    confusion_plot(yt_confusions['VADER 2014'],'VADER 2014: '+('row proportions' if normalise else 'counts'),ax,normalise)
close_figure(fig,'**Figure 5.** Primary replicated VADER, counts and reference-row-normalized proportions. Normalisation separates class recall from class prevalence; both views are needed when positive comments dominate.')

## 13. LLM versus each independent human; reference sensitivity

Agreement with a reference containing a human's own score is not a fair standalone measure of that human's accuracy. We therefore report **LLM versus annotator 1**, **LLM versus annotator 2**, and independent **human–human** agreement separately. This permits descriptive benchmarking on equivalent items without treating agreement with an average as the same estimand as inter-annotator reliability. No human-equivalence threshold or universal κ interpretation is used.

We also evaluate sign-only consensus items (both humans agree on polarity), the mean-reference full cohort and any actual adjudicated subset separately. Consensus-only selection excludes hard cases and can overstate performance; it is sensitivity evidence, not the replacement primary sample.

In [ ]:
individual_rows=[];individual_draws={}
for n in [1,2]:
    subset=yt.dropna(subset=[f'human_{n}_raw']).reset_index(drop=True)
    ix=bootstrap_indices(len(subset),subset.source_video_id)
    for system in SYSTEMS:
        m,_,_=evaluate(subset[f'human_{n}_label'],subset[system+'_label'],subset[f'human_{n}_intensity'],subset[system+'_score'])
        draws=bootstrap_evaluation(subset[f'human_{n}_label'],subset[system+'_label'],ix)
        individual_draws[(system,n)]=draws
        individual_rows.append(add_intervals({'system':system,'reference':f'Human {n}',**m},draws))
individual_human_results=pd.DataFrame(individual_rows)
display(individual_human_results[['system','reference','n','accuracy','macro_f1','cohen_kappa','cohen_kappa_ci_low','cohen_kappa_ci_high','pearson_r','mae']].round(4))
reference_sensitivity=[]
consensus=yt[yt.human_1_label.eq(yt.human_2_label)&yt.human_1_label.notna()]
for system in SYSTEMS:
    for name,sub,gold in [('mean full cohort',evaluation_yt,evaluation_yt.human_mean_label),
                         ('human polarity agreement only',consensus,consensus.human_1_label)]:
        m,_,_=evaluate(gold,sub[system+'_label'])
        reference_sensitivity.append({'system':system,'reference':name,**m})
    if adjudicated is not None and len(adjudicated):
        sub=yt.merge(adjudicated[['sample_id','human_gold_label']],on='sample_id',suffixes=('','_actual'),validate='one_to_one')
        m,_,_=evaluate(sub.human_gold_label_actual,sub[system+'_label'])
        reference_sensitivity.append({'system':system,'reference':'actual adjudicated subset',**m})
reference_sensitivity=pd.DataFrame(reference_sensitivity)
display(reference_sensitivity[['system','reference','n','accuracy','macro_f1','cohen_kappa']].round(4))
fig,ax=plt.subplots(figsize=(8.8,4.5))
llms=SYSTEMS[2:]
for i,(name,colour) in enumerate([('Human 1','#40576d'),('Human 2','#8b9b9f')]):
    d=individual_human_results[(individual_human_results.system.isin(llms))&(individual_human_results.reference==name)].set_index('system').loc[llms]
    xx=np.arange(3)+(i-.5)*.12
    ax.errorbar(xx,d.cohen_kappa,yerr=np.vstack([d.cohen_kappa-d.cohen_kappa_ci_low,d.cohen_kappa_ci_high-d.cohen_kappa]),fmt='o',capsize=3,label='LLM versus '+name,color=colour)
ax.axhline(HH['cohen_kappa'],color='gray',linestyle='--',label='Human–human κ (different comparison)')
ax.set(xticks=range(3),xticklabels=llms,xlabel='Saved LLM judge',ylabel='Unweighted polarity Cohen κ',title='LLM alignment to each human, with conditional 95% CIs')
ax.legend(fontsize=8)
close_figure(fig,'**Figure 6.** Source: individual human exports and saved LLM labels. The human–human line is descriptive, not an equivalence threshold. These dependent comparisons share items and annotators; overlap or non-overlap of marginal intervals is not a formal equivalence test.')

## 14. Pairwise agreements: LLM–LLM and VADER–LLM

All pairwise entries use common items and **unweighted categorical κ**, avoiding numeric encoding of nominal labels as continuous scores. The matrix includes humans and systems for context but is not a pooled multi-rater reliability coefficient. For VADER–LLM tables, classification metrics are directional relative to the named LLM comparison reference; they must not be described as validated VADER accuracy.

In [ ]:
raters={'Human 1':evaluation_yt.human_1_label.to_numpy(),'Human 2':evaluation_yt.human_2_label.to_numpy(),
        **{s:evaluation_yt[s+'_label'].to_numpy() for s in SYSTEMS}}
agreement_matrix=pd.DataFrame(index=raters,columns=raters,dtype=float)
pairwise_rows=[]
for left,right in itertools.combinations(raters,2):
    m,_,_=evaluate(raters[left],raters[right])
    agreement_matrix.loc[left,right]=agreement_matrix.loc[right,left]=m['cohen_kappa']
    pairwise_rows.append({'evaluator_1':left,'evaluator_2':right,'n':len(Y),'agreement':m['accuracy'],'cohen_kappa':m['cohen_kappa']})
for evaluator in raters: agreement_matrix.loc[evaluator,evaluator]=1
pairwise_agreement=pd.DataFrame(pairwise_rows)
display(pairwise_agreement[(pairwise_agreement.evaluator_1.isin(SYSTEMS[2:]))&(pairwise_agreement.evaluator_2.isin(SYSTEMS[2:]))].round(4))
vader_llm=[]
for v in SYSTEMS[:2]:
    for llm in SYSTEMS[2:]:
        m,_,_=evaluate(evaluation_yt[llm+'_label'],evaluation_yt[v+'_label'])
        vader_llm.append({'system':v,'comparison_reference':llm,**m})
vader_llm=pd.DataFrame(vader_llm)
display(vader_llm[['system','comparison_reference','n','accuracy','macro_f1','cohen_kappa']].rename(columns={'accuracy':'exact_agreement'}).round(4))
fig,ax=plt.subplots(figsize=(8,6.5))
sns.heatmap(agreement_matrix,annot=True,fmt='.2f',cmap='Blues',vmin=0,vmax=1,ax=ax)
ax.set(title=f'Pairwise polarity agreement: Cohen κ (n={len(Y)})',xlabel='Evaluator',ylabel='Evaluator')
close_figure(fig,'**Figure 7.** Source: aligned human and model labels. High LLM–LLM agreement may reflect shared conventions or shared errors; it does not prove correctness or independence. Human–human reliability remains a separate two-person analysis.')

## 15. Class prevalence, score behaviour and per-class performance

Class shares, uncertainty flags and score spreads describe observed outputs. They cannot establish a general provider bias or generation stability from one saved run. Category and feature associations are exploratory and may reflect different reference prevalence. No confidence score is treated as calibrated.

In [ ]:
distribution_inputs={'Human mean':evaluation_yt.human_mean_label,'Human 1':evaluation_yt.human_1_label,
                     'Human 2':evaluation_yt.human_2_label,**{s:evaluation_yt[s+'_label'] for s in SYSTEMS}}
label_distribution=pd.DataFrame({k:pd.Series(v).value_counts().reindex(LABELS,fill_value=0) for k,v in distribution_inputs.items()}).T
score_spread=pd.DataFrame([{'evaluator':s,'mean_score':evaluation_yt[s+'_score'].mean(),
    'sd_score':evaluation_yt[s+'_score'].std(ddof=0),'minimum':evaluation_yt[s+'_score'].min(),
    'maximum':evaluation_yt[s+'_score'].max(),'zero_share':evaluation_yt[s+'_score'].eq(0).mean()} for s in SYSTEMS]+
    [{'evaluator':'Human mean','mean_score':evaluation_yt.human_mean_intensity.mean(),'sd_score':evaluation_yt.human_mean_intensity.std(ddof=0),
      'minimum':evaluation_yt.human_mean_intensity.min(),'maximum':evaluation_yt.human_mean_intensity.max(),'zero_share':evaluation_yt.human_mean_intensity.eq(0).mean()}])
display(label_distribution)
display(score_spread.round(4))
fig,axes=plt.subplots(1,2,figsize=(11,4.5))
(label_distribution.div(label_distribution.sum(1),axis=0)*100).plot.bar(stacked=True,ax=axes[0],color=['#8d6a61','#b6bec4','#477e78'])
axes[0].set(ylabel='Share of labels (%)',xlabel='Evaluator / reference',title=f'YouTube polarity prevalence (n={len(Y)})')
axes[0].tick_params(axis='x',rotation=40)
pc=yt_per_class['VADER 2014'][['precision','recall','f1-score']]
pc.plot.bar(ax=axes[1],color=['#40576d','#7f969b','#af7139'])
axes[1].set(ylabel='Metric value (0–1)',xlabel='Human-reference class',ylim=(0,1),title='Primary VADER: class-specific performance')
axes[1].tick_params(axis='x',rotation=0)
close_figure(fig,'**Figure 8.** Source: the common YouTube cohort. Predicted prevalence and class-specific retrieval scores expose asymmetries hidden by aggregate accuracy. Differences are observed behaviours on this sample, not established universal model biases.')

## 16. Cross-dataset and within-YouTube comparisons

The same fixed classifier versions and declared class rules are compared, but reference provenance differs: original aggregate intensity labels, mapped multi-label emotions, and a two-person mean. Cross-dataset gaps cannot be assigned uniquely to domain, time, slang or emoji use. These are descriptive generalisation comparisons, not a causal temporal-drift experiment. GoEmotions test is used here, not its training or pooled descriptive results.

In [ ]:
cross_rows=[]
for _,r in original_results.iterrows():
    cross_rows.append({'dataset':r.dataset,'system':r.system,'n':r.n,'reference':'original aggregate intensity /4, +/-0.05',
        **{k:r[k] for k in ['accuracy','macro_f1','weighted_f1','pearson_r','macro_f1_ci_low','macro_f1_ci_high']}})
for _,r in go_results.query("split=='test'").iterrows():
    cross_rows.append({'dataset':'GoEmotions test','system':r.system,'n':r.n,'reference':'project-mapped original emotions',
        **{k:r[k] for k in ['accuracy','macro_f1','weighted_f1','macro_f1_ci_low','macro_f1_ci_high']}})
for _,r in master_results[master_results.system.isin(SYSTEMS[:2])].iterrows():
    cross_rows.append({'dataset':'YouTube','system':r.system,'n':r.n,'reference':r.reference,
        **{k:r[k] for k in ['accuracy','macro_f1','weighted_f1','pearson_r','macro_f1_ci_low','macro_f1_ci_high']}})
cross_dataset=pd.DataFrame(cross_rows)
display(cross_dataset.round(4))
all_frames={**originals,'GoEmotions test':go[go.original_split=='test'],'YouTube':evaluation_yt}
class_by_dataset=pd.DataFrame({k:v.human_gold_label.value_counts().reindex(LABELS,fill_value=0) for k,v in all_frames.items()}).T
size_table=pd.DataFrame({'dataset':list(all_frames),'n':[len(d) for d in all_frames.values()],
    'median_text_characters':[d.original_text.str.len().median() for d in all_frames.values()]})
display(size_table)
fig,axes=plt.subplots(1,2,figsize=(11,4.5))
axes[0].barh(size_table.dataset,size_table.n,color='#40576d')
axes[0].set(xlabel='Evaluation records (log scale)',ylabel='Dataset / reference',xscale='log',title='Evaluation sample sizes; GoEmotions test only')
(class_by_dataset.div(class_by_dataset.sum(1),axis=0)*100).plot.bar(stacked=True,ax=axes[1],color=['#8d6a61','#b6bec4','#477e78'])
axes[1].set(xlabel='Dataset / reference',ylabel='Reference label share (%)',title='Reference class distributions under declared rules')
axes[1].tick_params(axis='x',rotation=35)
close_figure(fig,'**Figure 9.** Sources: actual evaluation cohorts. The logarithmic size axis retains the small YouTube sample; class shares show that benchmarks differ in prevalence as well as language. Original emotion-label construction and human-reference differences limit direct comparability.')
fig,ax=plt.subplots(figsize=(9,4.8))
for offset,system in [(-.12,'VADER 2014'),(.12,'VADER 3.3.2')]:
    d=cross_dataset[cross_dataset.system==system]
    x=np.arange(len(d))+offset
    ax.errorbar(x,d.macro_f1,yerr=np.vstack([d.macro_f1-d.macro_f1_ci_low,d.macro_f1_ci_high-d.macro_f1]),fmt='o',capsize=3,label=system,color=PALETTE[system])
ax.set(xticks=np.arange(len(d)),xticklabels=d.dataset,xlabel='Dataset (different reference provenance)',ylabel='Macro F1 (0–1)',ylim=(0,1),title='VADER generalisation: macro F1 with conditional 95% CIs')
ax.legend(fontsize=9)
close_figure(fig,'**Figure 10.** Same system versions; differing reference constructs. CIs resample items from each fixed corpus and do not make the datasets exchangeable. The figure supports a performance profile, not proof of temporal drift.')

domain_rows=[]
for domain,sub in evaluation_yt.groupby('domain',sort=True):
    indices=bootstrap_indices(len(sub))
    for system in SYSTEMS:
        m,_,_=evaluate(sub.human_gold_label,sub[system+'_label'],sub.human_mean_intensity,sub[system+'_score'])
        draws=bootstrap_evaluation(sub.human_gold_label,sub[system+'_label'],indices)
        domain_rows.append(add_intervals({'domain':domain,'system':system,**m},draws))
domain_results=pd.DataFrame(domain_rows)
display(domain_results[['domain','system','n','accuracy','macro_f1','macro_f1_ci_low','macro_f1_ci_high','pearson_r']].round(4))
fig,ax=plt.subplots(figsize=(10,4.8))
domains=list(domain_results.domain.unique())
for i,system in enumerate(PRIMARY_SYSTEMS):
    d=domain_results[domain_results.system==system].set_index('domain').loc[domains]
    xx=np.arange(4)+(i-1.5)*.12
    ax.errorbar(xx,d.macro_f1,yerr=np.vstack([d.macro_f1-d.macro_f1_ci_low,d.macro_f1_ci_high-d.macro_f1]),fmt='o',capsize=2,label=system,color=PALETTE[system])
ax.set(xticks=range(4),xticklabels=[s.replace(' and ',' &\n') for s in domains],xlabel='Source-video category (one video per category)',ylabel='Macro F1 (0–1)',ylim=(0,1),title='YouTube within-video performance, n=50 per video')
ax.legend(fontsize=8)
close_figure(fig,'**Figure 11.** Source: fixed 50-comment allocations. These are within-video item-bootstrap intervals, with wide uncertainty on small classes. Category, creator and audience are confounded, so the plot cannot establish genre-level performance differences.')

## 17. Paired comparisons and uncertainty

The declared primary comparison family comprises the six pairs among historical VADER and the three LLMs. Δmacro-F1 intervals use identical video-stratified item draws; McNemar uses the observed paired correct/incorrect decisions. Exact two-sided McNemar is implemented as a binomial test on discordant pairs, with Holm-adjusted p-values across six tests. Modern-versus-historical VADER is a separately labelled version-sensitivity comparison, not added opportunistically to the primary family.

These tests are **conditional exploratory analyses**: within-video comment independence is not established, and neither bootstrap nor a small p-value corrects selection bias or an imperfect reference. No independent-sample test is used. Interval exclusion of zero and nominal significance are not a population claim or a practical-importance threshold. Leave-one-video-out results reveal sensitivity to the four observed sources rather than estimating an unseen-video population.

In [ ]:
paired_comparisons=[]
for left,right in itertools.combinations(PRIMARY_SYSTEMS,2):
    left_correct=evaluation_yt[left+'_label'].to_numpy()==Y
    right_correct=evaluation_yt[right+'_label'].to_numpy()==Y
    left_only=int(np.sum(left_correct&~right_correct));right_only=int(np.sum(~left_correct&right_correct))
    p=binomtest(left_only,left_only+right_only,p=.5,alternative='two-sided').pvalue if left_only+right_only else 1.
    diffs=yt_boot[left].macro_f1.to_numpy()-yt_boot[right].macro_f1.to_numpy()
    lo,hi,bad=percentile_ci(diffs)
    obs=master_results.set_index('system')
    paired_comparisons.append({'left':left,'right':right,'n':len(Y),'delta_accuracy':float(left_correct.mean()-right_correct.mean()),
        'delta_macro_f1':float(obs.loc[left,'macro_f1']-obs.loc[right,'macro_f1']),
        'delta_macro_f1_ci_low':lo,'delta_macro_f1_ci_high':hi,'left_only_correct':left_only,'right_only_correct':right_only,
        'mcnemar_exact_p':float(p),'degenerate_draws':bad})
paired_comparisons=pd.DataFrame(paired_comparisons)
paired_comparisons['mcnemar_holm_p']=multipletests(paired_comparisons.mcnemar_exact_p,method='holm')[1]
display(paired_comparisons.round(5))
fig,ax=plt.subplots(figsize=(9,4.8))
d=paired_comparisons.iloc[::-1].reset_index(drop=True)
ax.errorbar(d.delta_macro_f1,np.arange(len(d)),xerr=np.vstack([d.delta_macro_f1-d.delta_macro_f1_ci_low,d.delta_macro_f1_ci_high-d.delta_macro_f1]),fmt='o',capsize=3,color='#40576d')
ax.axvline(0,color='gray',linestyle='--')
ax.set(yticks=np.arange(len(d)),yticklabels=d.left+' minus '+d.right,xlabel='Paired macro-F1 difference (left − right)',ylabel='Matched system comparison',title=f'Conditional paired 95% bootstrap intervals (B={BOOTSTRAPS}, n={len(Y)})')
close_figure(fig,'**Figure 12.** Source: identical human-reference items and matched resamples. Positive differences favour the named left system on this reference. Interval width and the four-video sampling limitation matter; this is not an equivalence or population superiority test.')
version_delta=yt_boot['VADER 3.3.2'].macro_f1-yt_boot['VADER 2014'].macro_f1
print('Version-sensitivity Δmacro F1 CI (modern minus historical):',percentile_ci(version_delta)[:2])
loo=[]
for held_out,part in evaluation_yt.groupby('source_video_id'):
    sub=evaluation_yt[evaluation_yt.source_video_id!=held_out]
    for system in PRIMARY_SYSTEMS:
        m,_,_=evaluate(sub.human_gold_label,sub[system+'_label'])
        loo.append({'omitted_source_category':part.domain.iloc[0],'system':system,'n':len(sub),'accuracy':m['accuracy'],'macro_f1':m['macro_f1']})
leave_one_video_out=pd.DataFrame(loo)
display(leave_one_video_out.round(4))
# Numerical QA: deterministic resampling and paired-statistic identity.
assert np.array_equal(YT_INDICES,bootstrap_indices(len(evaluation_yt),evaluation_yt.source_video_id))
assert np.allclose(yt_boot['VADER 2014'].macro_f1-yt_boot['VADER 2014'].macro_f1,0)
print('Bootstrap index reproducibility and zero self-difference checks passed.')

## 18. Systematic disagreement and error audit

The audit classifies **observable prediction patterns**, not causes: historical VADER wrong/LLM right, the reverse, both wrong, all primary systems agree against the human reference, all evaluators agree, and human disagreement. These sets can overlap and are not summed as mutually exclusive categories. Full underlying audit data stay available in notebook variables.

A proposed causal coding scheme would allow multiple tags (sarcasm, slang, emoji, mixed sentiment, negation, intensifier, capitals, context dependence, lexical gap, neutral ambiguity), an evidence excerpt, coder ID and verification status. Without verified coding, no error-type-frequency chart is produced. String/Unicode indicators below are explicitly **screening proxies**, not verified causal categories. Any analyst-selected qualitative explanation generated during this notebook's preparation is provisional, not a human-coded research annotation.

In [ ]:
audit=evaluation_yt[['sample_id','domain','original_text','human_1_label','human_2_label','human_gold_label','human_1_raw','human_2_raw','human_mean_raw']].copy()
for system in SYSTEMS:
    audit[system+'_label']=evaluation_yt[system+'_label'].to_numpy()
    audit[system+'_correct']=audit[system+'_label']==audit.human_gold_label
pattern_rows=[]
for llm in SYSTEMS[2:]:
    v=audit['VADER 2014_correct'];l=audit[llm+'_correct']
    for pattern,mask in [('VADER wrong; LLM correct',~v&l),('VADER correct; LLM wrong',v&~l),('both wrong',~v&~l)]:
        pattern_rows.append({'LLM':llm,'pattern':pattern,'count':int(mask.sum())})
primary_cols=[s+'_label' for s in PRIMARY_SYSTEMS]
audit['all_primary_models_agree']=audit[primary_cols].nunique(axis=1).eq(1)
audit['all_models_agree_against_reference']=audit.all_primary_models_agree & audit['VADER 2014_label'].ne(audit.human_gold_label)
audit['humans_disagree']=audit.human_1_label.ne(audit.human_2_label)
audit['all_evaluators_agree']=audit[primary_cols+['human_1_label','human_2_label']].nunique(axis=1).eq(1)
display(pd.DataFrame(pattern_rows))
display(audit[['all_models_agree_against_reference','humans_disagree','all_evaluators_agree']].sum().rename('count').to_frame())
display(audit.loc[audit.all_models_agree_against_reference,['sample_id','domain','human_gold_label']+primary_cols])

feature_flags=pd.DataFrame(index=audit.index)
text=audit.original_text
feature_flags['negation token proxy']=text.str.contains(r"\b(?:not|never|no)\b|n't\b",case=False,regex=True)
feature_flags['emoji/symbol range proxy']=text.map(lambda t:any(0x1F300<=ord(c)<=0x1FAFF or 0x2600<=ord(c)<=0x27BF for c in t))
feature_flags['all-caps word proxy']=text.map(lambda t:any(len(w)>1 and w.isalpha() and w.isupper() for w in t.split()))
feature_flags['repeated punctuation proxy']=text.str.contains(r'[!?]{2,}',regex=True)
feature_rows=[]
for feature,flag in feature_flags.items():
    for value in [True,False]:
        subset=audit[flag.eq(value)]
        if len(subset):feature_rows.append({'screening_proxy':feature,'present':value,'n':len(subset),
            'VADER_2014_error_rate':float(1-subset['VADER 2014_correct'].mean())})
display(pd.DataFrame(feature_rows).round(4))
note('Proxy groups are not controlled experiments, are overlapping and may differ in sentiment prevalence. They identify records for inspection; they do not establish that the feature caused an error. Missing-context and sarcasm cannot be reliably inferred from punctuation alone.')

coding_path=ROOT/'data/youtube/annotations/youtube_error_coding.csv'
if coding_path.is_file():
    coding=load_csv(str(coding_path.relative_to(ROOT)))
    needed={'sample_id','error_tags','coder_id','verification_status'}
    assert needed<=set(coding.columns) and set(coding.sample_id)<=set(yt.sample_id)
    verified=coding[coding.verification_status.eq('human_verified') & coding.coder_id.str.strip().ne('')]
    if len(verified):
        tag_counts=verified.assign(tag=verified.error_tags.str.split(';')).explode('tag').query("tag!=''").tag.value_counts()
        display(tag_counts.rename('verified_count').to_frame())
        fig,ax=plt.subplots(figsize=(8,4));tag_counts.plot.barh(ax=ax,color='#40576d')
        ax.set(xlabel='Verified coded cases (multi-tag allowed)',ylabel='Human-verified error tag',title='Available verified qualitative codes')
        close_figure(fig,'Verified-code counts describe the supplied coding file; multi-tag counts overlap and are not causal effect estimates.')
    else:note('**Not computable from available data:** verified error-type-frequency plot; no human-verified coding rows.')
else:
    note('**Not computable from available data:** verified sarcasm/slang/context error-type frequencies. No human-coded error-category file was supplied.')

### 18.1 Redacted qualitative review

The following excerpts are local, bounded and redacted for URLs, handles, email and phone-like strings. They are shown to interpret observable disagreement, not to publish comment text or establish author intent. Examples are purposively selected, so they cannot estimate error-type prevalence. Human adjudication must remain separate from any model-visible notebook review. Existing ratings are never changed.

In [ ]:
def redacted_excerpt(text,limit=180):
    text=re.sub(r'https?://\S+','[URL]',str(text))
    text=re.sub(r'[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}','[EMAIL]',text)
    text=re.sub(r'(?<!\w)@[\w.]+','[HANDLE]',text)
    text=re.sub(r'(?<!\w)(?:\+?\d[\d ()-]{7,}\d)(?!\w)','[NUMBER]',text)
    return text[:limit]+('…' if len(text)>limit else '')
# Deterministic representative IDs, plus observed unanimous-error cases.
example_ids=[sid for sid in ['COM001','COM002','COM003','COM004','COM201','COM202'] if sid in set(audit.sample_id)]
example_ids=list(dict.fromkeys(example_ids+audit.loc[audit.all_models_agree_against_reference,'sample_id'].head(2).tolist()))
examples=audit.set_index('sample_id').loc[example_ids].reset_index()
examples['redacted_excerpt']=examples.original_text.map(redacted_excerpt)
examples['observed_pattern']=examples.apply(lambda r:'humans disagree' if r.humans_disagree else 'humans agree',axis=1)
display(examples[['sample_id','redacted_excerpt','human_1_raw','human_2_raw','human_gold_label']+primary_cols+['observed_pattern']])
note('**Provisional interpretive notes:** COM002 combines an apparently positive opening with an accusation; literal word valence may compete with intended criticism. COM003 combines derogatory wording and a laughing emoji, so neither the emoji nor a single sentiment word establishes intent. COM001 is an emphatic compliment with a crying emoji, illustrating why emoji meanings are contextual. These are candidate readings by an AI analyst, not independently verified causal codes. Compare the displayed actual predictions before claiming which system handled each case better. Unanimous model errors show why inter-model agreement cannot replace the human reference.')

## 19. Intensity concordance and residuals

The primary VADER compound score is nonlinear and not a calibrated estimate of human mean/4. The identity line and residual plot are therefore descriptive under the declared shared-range assumption; they distinguish correlation from agreement. Human means are discrete averages, so repeated x-values are expected. LLM score calibration cannot be fully interpreted without verified historical model/role/generation settings.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4.5))
h=evaluation_yt.human_mean_intensity.to_numpy();v=evaluation_yt['VADER 2014_score'].to_numpy()
axes[0].scatter(h,v,s=22,alpha=.45,color=PALETTE['VADER 2014'])
axes[0].plot([-1,1],[-1,1],color='gray',linestyle='--',label='Identity, not fitted')
axes[0].set(xlabel='Mean human raw score / 4 (−1 to +1)',ylabel='VADER 2014 compound (−1 to +1)',title=f'Primary VADER intensity versus human mean (n={len(h)})')
axes[0].legend(fontsize=8)
axes[1].scatter((h+v)/2,v-h,s=22,alpha=.45,color=PALETTE['VADER 2014'])
axes[1].axhline(0,color='gray',linestyle='--',label='Zero difference')
axes[1].axhline(np.mean(v-h),color='#af7139',label='Observed mean difference')
axes[1].set(xlabel='Mean of human-normalized and VADER scores',ylabel='VADER minus normalized human score',title='Descriptive intensity-difference plot')
axes[1].legend(fontsize=8)
close_figure(fig,'**Figure 13.** Source: the same current human/model pairs. Correlation concerns co-movement; residual displacement concerns score agreement. No clinical limits of agreement or calibration claim is made for these bounded, ordinal-derived scores. Differences reflect both model behaviour and reference construction.')

## 20. Consolidated findings and reporting tables

The tables below are computed from the current inputs. They provide results to transfer into the ACL report while preserving the distinctions between original replication, derived sentiment targets and the custom human reference. The final narrative is generated from calculated statistics rather than historical project notes. A best point estimate is not an unconditional claim of model superiority.

In [ ]:
print('MASTER RESULTS: common YouTube cohort')
display(master_results[['system','n','accuracy','macro_f1','weighted_f1','macro_precision','macro_recall','cohen_kappa',
    'accuracy_ci_low','accuracy_ci_high','macro_f1_ci_low','macro_f1_ci_high','pearson_r','spearman_rho','mae','rmse','ccc']].round(4))
print('HUMAN AGREEMENT')
display(human_agreement_table[['n','accuracy','cohen_kappa','cohen_kappa_ci_low','cohen_kappa_ci_high',
    'quadratic_weighted_kappa','quadratic_weighted_kappa_ci_low','quadratic_weighted_kappa_ci_high',
    'linear_weighted_kappa','pearson_r','pearson_r_ci_low','pearson_r_ci_high','spearman_rho','mae','rmse','ccc','ccc_ci_low','ccc_ci_high','raw_intensity_mae','exact_intensity_agreement']].round(4))
print('LLM VALIDATION AGAINST EACH HUMAN')
display(individual_human_results[individual_human_results.system.isin(SYSTEMS[2:])][['system','reference','n','accuracy','macro_f1','cohen_kappa','cohen_kappa_ci_low','cohen_kappa_ci_high']].round(4))
print('CROSS-DATASET VADER RESULTS: differing reference provenance')
display(cross_dataset.round(4))

best=master_results[master_results.system.isin(SYSTEMS[2:])].sort_values('macro_f1',ascending=False).iloc[0]
primary_v=master_results.set_index('system').loc['VADER 2014']
go_primary=go_results.query("split=='test' and system=='VADER 2014'").iloc[0]
tweet_primary=original_results.query("dataset=='Tweets' and system=='VADER 2014'").iloc[0]
findings=f"""
### Research findings

**RQ1 — Replication and generalisation.** Historical VADER obtains tweet r={tweet_primary.pearson_r:.4f}; comparison with each published corpus and source-artifact audit supports partial replication. Three-class F1 remains sensitive to reference thresholds and averaging. On the GoEmotions official test subset (n={int(go_primary.n)}), the same historical system obtains accuracy={go_primary.accuracy:.3f}, macro F1={go_primary.macro_f1:.3f}. On YouTube (n={int(primary_v.n)}), it obtains accuracy={primary_v.accuracy:.3f}, macro F1={primary_v.macro_f1:.3f} and r={primary_v.pearson_r:.3f}. These are different reference constructs, so their gaps do not isolate temporal drift.

**RQ2 — Human and LLM reliability.** Humans agree on {HH['accuracy']:.1%} of polarity labels (κ={HH['cohen_kappa']:.3f}); nine-level quadratic weighted κ={HH['quadratic_weighted_kappa']:.3f}. This reproduces the historical rounded agreement claims on the current matched ratings. {best.system} has the largest saved-LLM macro-F1 point estimate ({best.macro_f1:.3f}) against the current primary reference. Individual-human comparisons, paired intervals and prevalence must qualify that ranking. Missing exact model versions, prompt role placement, generation settings, blinding evidence and repeat judgments prevent a fully reproducible historical judge-generation study. These files can support a bounded empirical comparison; they are not validated universal replacements for humans.

**RQ3 — Comparative performance and limitations.** The paired-comparison table reports absolute accuracy/F1 effects, conditional confidence intervals and exploratory Holm-adjusted McNemar p-values. Observable disagreement patterns and short excerpts suggest hypotheses about lexical valence, ironic phrasing and emoji interpretation, but verified causal error categories are unavailable. Agreement among models can coexist with disagreement from the human reference. Four selected videos, quality amendments, ordinal-score averaging, possible within-source dependence and no independently adjudicated labels limit external validity. Small point-estimate differences should not be presented as a general model ranking.
"""
note(findings)

### 20.1 Ethics, publication and remaining evidence gaps

- Publicly visible comments are not a guarantee of consent or permission to redistribute text. Collection and retention constraints are documented locally; the unresolved cross-owner pooling/derived-use permission question is not certified as resolved by this notebook. This is a recorded project limitation, not a new legal determination.
- No commenter profiles or ages were collected. IDs and source metadata remain private. The notebook is stored under the existing lifecycle-controlled YouTube report directory and must be refreshed/deleted with its source snapshot, including exported copies and cached outputs. A public repository should contain runnable code and appropriately permitted inputs, not credentials or unrestricted API-derived text.
- The app instructed annotators to work independently without AI and to judge text alone. The files cannot independently verify participant age, consent, independence, reading behaviour or model-output blinding. Document those matters from actual research records; do not invent them.
- Original independent ratings are preserved. Model-visible analysis must not be represented as blind human adjudication. The averaged reference and cancellation-to-neutral behaviour are explicit limitations.
- No live model calls were made. To regenerate outputs later, obtain the actual protocol and use secure environment credentials; unknown historical settings cannot be reconstructed by guessing.
- The report should acknowledge AI assistance in implementation, literature checking and provisional qualitative interpretation. The researcher remains responsible for sources, methods and conclusions. Institution-specific ethics/lecture requirements still need checking against the supplied unit material.

**Not computable from available data:** actual adjudicated-reference results (unless an eligible optional file is supplied), repeat-run judge instability, verified causal error-type prevalence, participant demographics/independence, a true population-level video-cluster confidence interval, and exact regeneration of undocumented LLM judgments. These omissions are substantive limitations, not blank charts.

## 21. References and final reproducibility checks

Publication details and relevant claims were checked against primary publisher/ACL/arXiv/PubMed sources on **5 November 2026**; the original VADER publisher page blocked automated access, so its supplied local paper was used. Sources establish methodological context, not blanket validation of this project's annotation or sampling choices.

1. Hutto, C. J., and Gilbert, E. 2014. *VADER: A Parsimonious Rule-Based Model for Sentiment Analysis of Social Media Text.* ICWSM 8(1), 216–225. [DOI](https://doi.org/10.1609/icwsm.v8i1.14550). [Author software and score conventions](https://github.com/cjhutto/vaderSentiment#about-the-scoring).
2. Demszky, D., Movshovitz-Attias, D., Ko, J., Cowen, A., Nemade, G., and Ravi, S. 2020. *GoEmotions: A Dataset of Fine-Grained Emotions.* ACL, 4040–4054. [Paper](https://aclanthology.org/2020.acl-main.372/).
3. Artstein, R., and Poesio, M. 2008. *Inter-Coder Agreement for Computational Linguistics.* Computational Linguistics 34(4), 555–596. [Paper](https://aclanthology.org/J08-4004/).
4. Cohen, J. 1960. *A Coefficient of Agreement for Nominal Scales.* Educational and Psychological Measurement 20(1), 37–46. [DOI](https://doi.org/10.1177/001316446002000104).
5. Cohen, J. 1968. *Weighted Kappa: Nominal Scale Agreement with Provision for Scaled Disagreement or Partial Credit.* Psychological Bulletin 70(4), 213–220. [DOI](https://doi.org/10.1037/h0026256).
6. Zheng, L., et al. 2023. *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena.* NeurIPS Datasets and Benchmarks. [Author paper](https://arxiv.org/abs/2306.05685).
7. Liu, Y., Iter, D., Xu, Y., Wang, S., Xu, R., and Zhu, C. 2023. *G-Eval: NLG Evaluation using GPT-4 with Better Human Alignment.* EMNLP, 2511–2522. [Paper](https://aclanthology.org/2023.emnlp-main.153/).
8. Bavaresco, A., et al. 2025. *LLMs instead of Human Judges? A Large Scale Empirical Study across 20 NLP Evaluation Tasks.* ACL Volume 2: Short Papers, 238–255. [Paper](https://aclanthology.org/2025.acl-short.20/).
9. Dror, R., Baumer, G., Shlomov, S., and Reichart, R. 2018. *The Hitchhiker's Guide to Testing Statistical Significance in Natural Language Processing.* ACL, 1383–1392. [Paper](https://aclanthology.org/P18-1128/).
10. Lin, L. I. 1989. *A Concordance Correlation Coefficient to Evaluate Reproducibility.* Biometrics 45(1), 255–268. [PubMed](https://pubmed.ncbi.nlm.nih.gov/2720055/).

Pearson/Spearman and error functions are implemented through SciPy/NumPy, classification checks through scikit-learn, and Holm correction through statsmodels. Exact formulas, assumptions and executable numerical checks above define the project implementation. No result is inferred from a citation alone.

In [ ]:
# Critical-result checks independently reproduce persisted earlier results where methods match.
checks=[]
old_go_path=ROOT/'reports/vader_evaluation/goemotions_metrics.json'
if old_go_path.is_file():
    old=json.loads(old_go_path.read_text())['test']
    current=go_results.query("split=='test' and system=='VADER 3.3.2'").iloc[0]
    checks.append({'check':'modern GoEmotions test vs previous run','passed':bool(np.isclose(current.accuracy,old['accuracy']) and np.isclose(current.macro_f1,old['macro_f1']))})
old_y_path=ROOT/'reports/youtube/vader_evaluation/metrics.json'
if old_y_path.is_file():
    old=json.loads(old_y_path.read_text())
    if old.get('paired_annotation_count')==len(evaluation_yt):
        previous=old['references']['paired_human_mean']
        current=master_results.set_index('system').loc['VADER 3.3.2']
        checks.append({'check':'modern YouTube mean-reference vs previous run','passed':bool(np.isclose(current.accuracy,previous['accuracy']) and np.isclose(current.macro_f1,previous['macro_f1']))})
for system in SYSTEMS:
    pred=evaluation_yt[system+'_label'].to_numpy()
    cm=yt_confusions[system]
    row=master_results.set_index('system').loc[system]
    assert cm.sum()==len(Y)
    assert np.isclose(np.trace(cm)/cm.sum(),row.accuracy)
    assert np.isclose(f1_score(Y,pred,labels=LABELS,average='macro',zero_division=0),row.macro_f1)
checks.append({'check':'all YouTube confusion totals and independent sklearn recomputation','passed':True})
unchanged=all((ROOT/p).is_file() and sha(ROOT/p)==h for p,h in INPUT_HASHES.items())
checks.append({'check':'all tracked input bytes unchanged','passed':unchanged})
checks.append({'check':'all common-cohort joins one-to-one; no missing predictions','passed':len(evaluation_yt)==200 and evaluation_yt.sample_id.is_unique})
# Independent comparison with the portable modular pipeline, if already run.
portable_go=ROOT/'reports/goemotions/metrics.csv'
if portable_go.is_file():
    reference=pd.read_csv(portable_go)
    for system in SYSTEMS[:2]:
        before=reference[(reference['split']=='test')&(reference.system==system)].iloc[0]
        now=go_results[(go_results['split']=='test')&(go_results.system==system)].iloc[0]
        checks.append({'check':system+' GoEmotions vs standalone scripts','passed':bool(np.isclose(before.accuracy,now.accuracy) and np.isclose(before.macro_f1,now.macro_f1))})
portable_youtube=ROOT/'reports/youtube/model_metrics.csv'
if portable_youtube.is_file():
    reference=pd.read_csv(portable_youtube).set_index('system')
    for system in SYSTEMS:
        now=master_results.set_index('system').loc[system]
        checks.append({'check':system+' YouTube vs standalone scripts','passed':bool(all(np.isclose(reference.loc[system,k],now[k]) for k in ['accuracy','macro_f1','cohen_kappa','pearson_r']))})
qa=pd.DataFrame(checks)
display(qa)
assert qa.passed.all(),'An independent QA check failed; investigate before reporting'
input_hash_manifest=pd.DataFrame([{'path':p,'sha256':h} for p,h in sorted(INPUT_HASHES.items())])
print('Tracked input files:',len(input_hash_manifest))
display(input_hash_manifest)
print('Modern implementation artifact hashes:',MODEL_HASHES)
print('All analysis is inline. No raw data, annotations, labels or credentials were changed.')
print('Execution completed UTC:',datetime.now(timezone.utc).isoformat())